# Quant Alpha Research: Comparing XGBoost, Transformers, HMM Regimes, and Reinforcement Learning

## Project Overview

This project compares classical machine-learning methods, transformer-based sequence models, hidden Markov model regime detection, and reinforcement-learning trading policies for detecting and trading short-term market alpha.

The project does not assume that transformers, HMMs, or reinforcement learning will automatically outperform simpler models. The purpose is to determine whether each additional layer of complexity produces measurable out-of-sample improvement over a strong XGBoost baseline.

The first version focuses on predicting meaningful intraday NVDA price movements using synchronized five-minute market data.

---

# Research Objective

The primary research question is:

**Can transformer models, HMM-based regime detection, and reinforcement-learning-based trading policies improve predictive and trading performance compared with a strong XGBoost baseline?**

The systems will be evaluated using:

* Classification performance
* Per-class precision, recall, and F1 score
* Macro F1 score
* Probability calibration
* Log loss
* Brier score
* Trading PnL
* Transaction-cost-adjusted returns
* Sharpe ratio
* Maximum drawdown
* Turnover
* Out-of-sample stability

The project will separately test whether improvements come from:

* Transformer sequence modeling
* Volume and activity features
* Context assets
* Time-of-day and weekday features
* HMM regime features
* Reinforcement-learning trade management

---

# Assets

## Primary Trading Target

* NVDA

## Context Assets

* QQQ
* SPY

NVDA is the asset being predicted and traded.

QQQ and SPY are included as context assets to help distinguish:

* NVDA-specific movement
* Technology-market movement
* Broad-market movement

Additional assets may be tested later as separate ablation experiments, but the first version uses only NVDA, QQQ, and SPY.

---

# Data Source and Trading Timeline

The first version uses Alpaca SIP five-minute candles.

## Base Configuration

* Data source: Alpaca SIP
* Candle size: 5 minutes
* Primary target: NVDA
* Context assets: QQQ and SPY
* Trading style: intraday
* Prediction horizon: 12 candles
* Prediction window: 60 minutes
* Overnight positions: disabled
* Regular market open: 6:30 AM Pacific Time
* Regular market close: 1:00 PM Pacific Time
* Final regular-session candle start: 12:55 PM Pacific Time
* Daily trading-state reset: enabled

Each complete trading day contains:

* 21 pre-open context candles
* 78 regular-session candles
* 99 total aligned candles per asset

The pre-open context period begins approximately 105 minutes before the regular market open.

---

# Pre-Open Context Candles

The dataset retains 21 five-minute candles before the regular market open.

These candles are no longer treated as EMA warmup candles because EMA features are excluded from the first model version.

Instead, the pre-open candles are used as historical market context for:

* Transformer input sequences
* Previous-candle return calculations
* Previous-candle volume changes
* Price-action history
* Cross-asset history

The model is not allowed to trade during the pre-open context period.

Pre-open rows:

* May appear inside transformer input sequences
* May be used to calculate causal historical features
* Do not receive supervised prediction labels
* Do not receive trading rewards
* Do not create trades

The existing code may still label these rows with:

```text
label_status = warmup
```

Conceptually, these rows represent:

```text
pre-open context
```

rather than EMA warmup.

---

# Prediction Target

The first version frames the problem as a three-class triple-barrier classification task.

At the close of each valid regular-session candle `t`, the model predicts whether NVDA will hit a positive or negative one-percent barrier during the next 12 candles.

The model outputs:

* `P(neutral)`
* `P(up)`
* `P(down)`

## Execution and Reference Price

The model observes all information through the close of candle `t`.

Because the signal is generated after candle `t` closes, the earliest realistic execution price is:

```text
NVDA open at candle t+1
```

The triple-barrier reference price is therefore the next candle’s open.

For each prediction candle `t`:

```text
entry_price = NVDA open at t+1
upper_barrier = entry_price × 1.01
lower_barrier = entry_price × 0.99
```

The future evaluation window consists of:

```text
candles t+1 through t+12
```

## Label Definitions

### Up

The upper one-percent barrier is reached before the lower barrier.

### Down

The lower one-percent barrier is reached before the upper barrier.

### Neutral

Neither barrier is reached during the following 12 candles.

### Ambiguous Same-Bar Event

If the upper and lower barriers are both reached for the first time inside the same five-minute candle, the ordering cannot be determined from OHLC data.

These rows receive:

```text
label_status = ambiguous_same_bar
```

They do not receive a target class.

## Class-ID Mapping

The current classification mapping is:

```text
neutral = 0
up      = 1
down    = 2
```

These values are category identifiers only. They do not imply a numerical relationship between the classes.

---

# Handling Ambiguous Labels

Ambiguous rows are not included as supervised prediction targets because their correct directional class is unknown.

No classification loss is calculated for an ambiguous target row.

However, the underlying OHLCV market data remains valid. Therefore, an ambiguous candle may still appear inside the historical input sequence for a later valid prediction target.

For example:

```text
Candle 30 target: ambiguous
Candle 35 target: valid up
```

The sequence ending at candle 35 may include candle 30’s market features.

That is valid because:

* Candle 30’s OHLCV data is known
* Candle 30’s target label is not used as an input feature
* Loss is calculated only against candle 35’s valid target

Supervised model training uses only rows where:

```text
label_status = valid
```

---

# Current Labeled Dataset Status

The current aligned and labeled dataset contains:

* Complete trading days: 1,539
* Total rows: 152,361
* Total labelable regular-session rows: 101,574
* Valid supervised targets: 101,535
* Ambiguous same-bar targets: 39
* Pre-open context rows: 32,319
* Incomplete future-window rows: 18,468

## Current Class Distribution

Among valid targets:

* Neutral: 64,964 rows, approximately 63.98%
* Down: 19,100 rows, approximately 18.81%
* Up: 17,471 rows, approximately 17.21%

The ambiguous rate is extremely low, so ambiguous target rows can be excluded without materially reducing the dataset.

Entire days containing ambiguous rows will not be removed because the market data on those days remains valid.

---

# Data Cleaning and Session Validation

Before labeling, feature creation, training, or backtesting, the pipeline validates each trading day.

A day is retained only when every required asset has all expected aligned candles.

## Required Assets in Version One

* NVDA
* QQQ
* SPY

## Complete-Day Requirements

Each asset must have:

* 21 pre-open context candles
* 78 regular-session candles
* 99 total candles
* No duplicate timestamps
* No missing required values
* Identical aligned timestamps across assets

Days that fail these checks are removed before alignment.

The aligned dataset is created using exact timestamp matching across NVDA, QQQ, and SPY.

---

# Labeling Restrictions

The final 12 regular-session candles of each day cannot receive new supervised labels because their full future 60-minute windows would extend beyond the regular trading session.

These rows receive:

```text
label_status = incomplete_future_window
```

The first version does not permit:

* Overnight labels
* Overnight trades
* Prediction windows crossing into the next trading day
* Labels based on incomplete future windows

---

# Feature Design Principles

The first feature set focuses on normalized, causal, and relatively stationary information.

Raw price levels will not be used directly as model inputs because NVDA’s absolute price changes substantially across the historical period.

All features must be calculable using information available at or before the close of the current candle.

Future information may be used only to construct labels, never to construct model inputs.

If any scalers, averages, medians, or normalization statistics are used, they must be fitted using training data only.

Validation and test data must use the values fitted on the training period.

---

# No EMA Features in Version One

EMA features are excluded from the first version.

The available dataset begins only 21 candles before the regular market open. Although an EMA could technically be calculated, the earliest pre-open EMA values would be poorly initialized because earlier historical candles are unavailable.

Feeding those inconsistent EMA values into transformer sequences could introduce undesirable artifacts.

The first version will therefore not include:

* EMA 9
* EMA 20
* EMA crossovers
* EMA slopes
* EMA distances
* EMA spread
* Bars since EMA crossover

This decision allows all transformer input candles to use a consistent and reliable feature set.

EMA features may be reconsidered only as a later optional experiment if a reliable initialization method becomes available.

---

# Core Per-Candle Feature Set

Both XGBoost and the transformer will use the same underlying per-candle market information.

## Candle and Price-Action Features

For NVDA, QQQ, and SPY, useful normalized candle features include:

```text
log_return_1
body_pct
range_pct
upper_wick_pct
lower_wick_pct
close_position_in_range
vwap_distance
```

Example definitions:

```text
log_return_1 = log(close_t / close_t-1)

body_pct = (close - open) / open

range_pct = (high - low) / open

upper_wick_pct =
    (high - max(open, close)) / open

lower_wick_pct =
    (min(open, close) - low) / open

close_position_in_range =
    (close - low) / (high - low)

vwap_distance =
    (close - vwap) / close
```

A safe fallback must be used when the candle range is zero.

---

# Volume and Market-Activity Features

The first version will be volume-aware rather than purely volume-based.

Volume alone may indicate unusual activity but usually does not determine price direction. It will therefore be combined with price-action and cross-asset information.

Core volume and activity features include:

```text
log_volume
log_trade_count
average_trade_size
dollar_volume
log_dollar_volume
volume_change_1
trade_count_change_1
```

Example definitions:

```text
log_volume = log(1 + volume)

log_trade_count = log(1 + trade_count)

average_trade_size =
    volume / max(trade_count, 1)

dollar_volume =
    volume × vwap

log_dollar_volume =
    log(1 + dollar_volume)

volume_change_1 =
    log((volume_t + 1) / (volume_t-1 + 1))
```

Log transformations help reduce the extreme skew found in raw volume and trade-count data.

---

# Time-Adjusted Relative Volume

Volume follows a strong intraday pattern.

Volume is normally high near the market open, lower around midday, and often higher near the close.

Because of this, raw volume should not be interpreted identically at every time of day.

A useful feature is:

```text
time_adjusted_relative_volume =
    current volume
    divided by
    typical training-period volume for the same asset and time of day
```

For example:

```text
NVDA volume at 6:35 AM
÷
median NVDA 6:35 AM volume from the training period
```

The time-of-day volume baseline must be calculated using training data only.

Validation and test data must use the training-fitted baseline.

This prevents future information from leaking into the model.

---

# Cross-Asset Context Features

QQQ and SPY help determine whether an NVDA move is stock-specific or part of a broader market move.

Useful context features include:

```text
NVDA log return
QQQ log return
SPY log return

NVDA return minus QQQ return
NVDA return minus SPY return

QQQ return minus SPY return
```

The model may also compare time-adjusted activity levels across assets after each asset has been independently normalized.

Raw volume differences between NVDA, QQQ, and SPY should generally not be used because each asset naturally trades at a different scale.

---

# Time-of-Day Features

The behavior of the market changes throughout the trading day.

The model will receive a normalized time-of-day feature based on the number of minutes since the regular market open.

```text
minutes_since_open =
    candle time in minutes
    minus
    6:30 AM in minutes
```

Examples:

```text
4:45 AM  → -105 minutes
6:30 AM  → 0 minutes
8:10 AM  → 100 minutes
12:55 PM → 385 minutes
```

The normalized feature is:

```text
time_of_day =
    minutes_since_open / 390
```

Premarket values are negative.

Regular-session values range from approximately zero to one.

A linear time-of-day feature is preferred initially because the start and end of the regular session should not necessarily be treated as adjacent.

The raw absolute timestamp and raw calendar date will not be fed into the models.

---

# Day-of-Week Features

The model will also receive the weekday.

Weekday values will be one-hot encoded:

```text
weekday_monday
weekday_tuesday
weekday_wednesday
weekday_thursday
weekday_friday
```

The raw weekday integer will not be used directly because it could incorrectly suggest that Friday is numerically greater than Monday.

All candles from the same trading day will share the same weekday values.

---

# Historical Information and Model Fairness

The transformer and XGBoost will use the same underlying market information but represent history differently.

## Transformer Representation

The transformer receives an ordered fixed-length sequence of per-candle feature rows.

Initial candidate sequence length:

```text
20 candles
```

Each prediction sequence ends at the current candle `t`.

For a target at candle `t`:

```text
Input:
the previous 19 candles plus candle t

Target:
the triple-barrier label attached to candle t
```

Because the dataset includes 21 pre-open context candles, the transformer can create a full sequence for predictions beginning at the regular market open.

A prediction based on the 6:30 AM candle is generated after that candle closes and may execute at the 6:35 AM open.

## XGBoost Representation

XGBoost receives tabular features for the current prediction point.

To create a strong baseline, XGBoost may also receive causal lagged summaries derived from the same historical window available to the transformer.

Possible XGBoost lagged features include:

```text
return lag 1
return lag 3
return lag 6
return lag 12
cumulative return over recent candles
volume change lags
recent maximum range
recent average log volume
```

This gives XGBoost access to comparable historical information without giving it future data.

---

# Transformer Target Selection and Loss

Not every row becomes a transformer training target.

A transformer sequence is included as a supervised training example only when:

* The target row is during the regular session
* The target row has `label_status = valid`
* A complete input sequence exists
* Every feature in the sequence is available
* The sequence does not cross into a different trading day

Loss is calculated only against valid target labels.

Examples:

```text
Sequence ending on valid up target
→ included, calculate loss against up

Sequence ending on ambiguous target
→ excluded, no loss calculated

Sequence ending on incomplete future-window target
→ excluded, no loss calculated

Sequence ending on pre-open target
→ excluded, no loss calculated
```

An ambiguous or non-target candle may still appear inside the historical sequence for a later valid target.

---

# Class Imbalance

The current target distribution contains a majority neutral class.

Accuracy alone is not sufficient because a model could obtain high accuracy by frequently predicting neutral.

The models will be evaluated using:

* Confusion matrix
* Per-class precision
* Per-class recall
* Per-class F1 score
* Macro F1 score
* Log loss
* Brier score
* Probability calibration
* Performance on up and down events specifically

If necessary, the project may test:

* Class-weighted loss
* Balanced sampling
* Validation-selected probability thresholds
* Focal loss for the transformer

All class-balancing decisions must be selected using training and validation data only.

---

# Probability Calibration

Because the models output probabilities, the reliability of those probabilities is important.

A model predicting:

```text
P(up) = 0.70
```

should ideally be correct close to 70 percent of the time across similar predictions.

Calibration will be evaluated using:

* Brier score
* Log loss
* Reliability curves
* Confidence buckets
* Expected calibration error

Calibrated probabilities are important for:

* Trade filtering
* Position sizing
* Rule-based thresholds
* Reinforcement-learning state inputs

---

# Methods

## 1. XGBoost Baseline

XGBoost is the primary classical machine-learning baseline.

It will use:

* Normalized candle features
* Volume and activity features
* Time-adjusted relative volume
* QQQ and SPY context features
* Time-of-day features
* Day-of-week features
* Causal lagged and summary features
* Optional HMM regime features

The XGBoost classifier outputs:

* `P(neutral)`
* `P(up)`
* `P(down)`

The XGBoost model establishes the performance level that more complex methods must beat.

---

## 2. Transformer-Based Predictor

The transformer learns from ordered sequences of recent five-minute candles.

Each sequence contains the same core per-candle market features used by the XGBoost pipeline.

The transformer does not receive EMA features in version one.

The purpose of the transformer is to test whether it can learn useful temporal relationships from:

* Recent price action
* Volume evolution
* Trade-count activity
* Relative movement across assets
* Intraday timing
* Weekday effects

The transformer outputs:

* `P(neutral)`
* `P(up)`
* `P(down)`

---

## 3. Hidden Markov Model Regime Detection

An HMM will be tested as an optional regime-feature generator.

The HMM will not replace XGBoost or the transformer.

It will attempt to identify latent market conditions such as:

* Trending
* Choppy
* High activity
* Low activity
* Risk-on
* Risk-off

Possible HMM inputs include:

* NVDA return
* QQQ return
* SPY return
* NVDA relative return versus QQQ
* NVDA relative return versus SPY
* Candle range percentage
* Log volume
* Time-adjusted relative volume
* Trade-count activity

Possible HMM outputs include:

```text
hmm_regime
hmm_regime_prob_0
hmm_regime_prob_1
hmm_regime_prob_2
regime_changed
bars_since_regime_change
```

The HMM must be trained only on the training period.

Validation and test regimes must be inferred using the training-fitted HMM without refitting on future data.

---

## 4. Signal-State Features

After a prediction model outputs probabilities, the trading layer may create signal-state features such as:

```text
p_up
p_down
p_neutral
p_edge
p_abs_edge
signal_direction
signal_age_bars
signal_active
p_edge_change_1
p_edge_rolling_mean
signal_persistence
```

Example:

```text
p_edge = p_up - p_down
```

These features allow the trading system to distinguish between:

* Strong and weak predictions
* Fresh and stale signals
* Persistent and temporary signals
* Increasing and decreasing confidence

---

## 5. Rule-Based Trading Layer

Before reinforcement learning, the project will evaluate simple probability-based trading rules.

Example rules include:

* Go long when `P(up)` exceeds a threshold and sufficiently exceeds `P(down)`
* Go short when `P(down)` exceeds a threshold and sufficiently exceeds `P(up)`
* Stay flat when confidence is weak
* Stay flat when probabilities are too close
* Avoid new entries too close to market close

Thresholds will be tuned using validation data only.

The final test set will not be used for rule selection or threshold tuning.

---

## 6. Reinforcement-Learning Trading Agent

The reinforcement-learning agent acts as the trading decision layer.

It does not replace the prediction models.

The RL agent receives:

* Market features
* Prediction probabilities
* Signal-state features
* Optional HMM regime features
* Portfolio-state features

## RL Market-State Features

Possible market-state inputs include:

* NVDA price-action features
* NVDA volume and activity features
* QQQ context features
* SPY context features
* Time-adjusted relative volume
* Time of day
* Day of week

## RL Model-Signal Features

* `p_neutral`
* `p_up`
* `p_down`
* `p_edge`
* Signal direction
* Signal persistence
* Signal age
* Probability momentum

## RL Portfolio-State Features

* Current position
* Entry price
* Bars in position
* Unrealized PnL
* Realized PnL
* Maximum favorable excursion
* Maximum adverse excursion
* Remaining bars until market close

The purpose of RL is to learn:

* Whether to act on a signal
* When to enter
* When to exit
* When to remain flat
* How to manage risk
* How to handle persistent or changing signals

---

# RL Episode Structure

Each RL episode represents one complete trading day.

At the start of each day:

* Position resets to flat
* Entry price resets
* Unrealized PnL resets
* Realized daily PnL resets
* Bars in position reset
* Signal-state counters reset

The agent may observe pre-open context candles but cannot trade during them.

Trading begins during the regular session.

At the end of the regular trading day:

* Any remaining position is closed
* No position is carried overnight
* The episode terminates

---

# RL Action Space

The initial RL environment will use a discrete target-position action space:

```text
0 = short
1 = flat
2 = long
```

Later experiments may test continuous position sizing.

---

# RL Reward Function

The first RL reward function will prioritize transaction-cost-adjusted changes in portfolio value.

Possible reward components include:

* Realized PnL
* Unrealized PnL change
* Slippage
* Fees
* Spread costs
* Turnover penalty
* Drawdown penalty
* Penalty for holding positions near market close

Complex reward shaping will be added only after the basic RL environment is working correctly.

---

# Execution Timing

To avoid lookahead bias:

* Candle `t` completes
* Features through candle `t` become available
* Prediction is generated after candle `t` closes
* Trading decision is generated after candle `t` closes
* Earliest execution occurs at the open of candle `t+1`

The model may not use candle `t+1` as an input feature when predicting candle `t`.

The next candle’s open may be used during label construction because labels describe future outcomes.

This distinction is essential:

```text
Future data in the label:
allowed

Future data in model features:
not allowed
```

---

# Costs and Slippage

Backtests will include realistic estimated trading costs.

The first version will model:

* Slippage
* Fees or commissions
* Optional spread costs
* Position-size constraints
* Turnover

Example conservative execution assumptions:

```text
Long entry:
next open adjusted upward by slippage

Long exit:
execution price adjusted downward by slippage

Short entry:
next open adjusted downward by slippage

Short exit:
execution price adjusted upward by slippage
```

Primary reported performance will be transaction-cost adjusted.

---

# Validation and Test Methodology

The dataset will be split chronologically.

The final test period must remain locked until the methodology is finalized.

Because the available dataset may not contain exactly ten full years, the split will be based on the actual available date range rather than assuming a fixed number of years.

## Recommended Structure

* First portion of data: model development and walk-forward validation
* Final approximately 20 percent of dates: locked out-of-sample test period

Within the development period, expanding or rolling walk-forward folds will be used.

For every fold:

* Feature statistics are fitted on training data only
* Time-of-day volume baselines are fitted on training data only
* Scalers are fitted on training data only
* HMM is fitted on training data only
* XGBoost and transformer models are trained on training data only
* Validation data is used for tuning and model selection

## Purging and Embargo

Because neighboring target rows use overlapping future 12-candle windows, rows near train-validation or validation-test boundaries may share future information.

A purge or embargo of at least 12 candles should be applied around chronological split boundaries.

This helps prevent information from overlapping across dataset partitions.

---

# Model Selection and Hyperparameter Tuning

The final test set will not be used for:

* Feature selection
* Hyperparameter tuning
* Sequence-length selection
* Probability-threshold tuning
* Class-weight selection
* HMM regime-count selection
* RL reward tuning
* RL policy selection
* Backtest rule changes

Only after the methodology is locked will the final test set be evaluated.

---

# Non-ML Baselines

The project will compare the machine-learning systems against simple non-ML strategies.

Possible baselines include:

* Always flat
* Buy and hold NVDA during the test period
* Simple short-term momentum
* Simple short-term mean reversion
* Volume-breakout strategy
* Relative-volume strategy
* Random trading policy

These baselines help determine whether machine learning adds value beyond simple behavior.

---

# Ablation Studies

Ablation studies will determine which components create real improvement.

Important comparisons include:

* Without context assets versus with QQQ and SPY
* Without volume features versus with volume features
* Without time-adjusted relative volume versus with it
* Without time-of-day features versus with them
* Without weekday features versus with them
* Without HMM regime features versus with them
* XGBoost versus transformer
* Rule-based trading versus RL trading
* Different transformer sequence lengths
* Current-row XGBoost versus lag-enhanced XGBoost

EMA features are not part of the first-version ablation plan.

---

# Risk-Management Constraints

The backtester and RL environment will include explicit risk controls.

Initial controls may include:

* Maximum position size
* No overnight positions
* Forced flat before market close
* Maximum holding period
* Slippage and fee modeling
* Optional stop-loss
* Optional take-profit
* Optional maximum daily loss
* No new positions when insufficient trading time remains

---

# Main Model Comparison

## Without HMM Regime Features

1. XGBoost probabilities plus rule-based trading
2. Transformer probabilities plus rule-based trading
3. XGBoost probabilities plus RL trading agent
4. Transformer probabilities plus RL trading agent

## With HMM Regime Features

5. XGBoost plus HMM features plus rule-based trading
6. Transformer plus HMM features plus rule-based trading
7. XGBoost plus HMM features plus RL trading agent
8. Transformer plus HMM features plus RL trading agent

This structure answers:

1. Does the transformer improve prediction compared with XGBoost?
2. Does RL improve trading decisions compared with simple rules?
3. Do HMM regime features improve prediction or trading performance?

---

# Backtesting Assumptions

The first backtester will use the following explicit assumptions:

* Five-minute candles
* Intraday trading only
* No overnight positions
* Daily trading-state reset
* Pre-open candles used only as context
* No pre-open trading
* No pre-open supervised targets
* No new target when the complete future window is unavailable
* Signal generated after candle `t`
* Earliest execution at candle `t+1` open
* Slippage included
* Fees included
* Optional spread cost included
* Position limits included
* Forced flat before market close
* Chronological train, validation, and test splits
* Training-only fitting for all learned preprocessing statistics
* Ambiguous rows excluded as targets but retained as historical market context

---

# Reproducibility

Every experiment should record:

* Dataset version
* Included assets
* Feature configuration
* Sequence length
* Model architecture
* Hyperparameters
* Random seed
* Train-validation-test dates
* Purge and embargo settings
* Class weights
* Probability thresholds
* Backtest settings
* Slippage assumptions
* Fee assumptions
* Performance metrics

Every reported result should be reproducible from a saved configuration.

---

# Updated Project Workflow

1. Download or load Alpaca SIP candle data
2. Convert timestamps to Pacific Time
3. Retain 21 pre-open and 78 regular-session candles
4. Validate complete days for NVDA, QQQ, and SPY
5. Keep only dates complete across every required asset
6. Align assets on exact timestamps
7. Build triple-barrier labels using next-candle open as the reference price
8. Mark ambiguous same-bar targets
9. Mark final 12 candles as incomplete future-window targets
10. Save the complete labeled dataset
11. Build normalized candle features
12. Build volume and activity features
13. Build time-adjusted relative-volume features using training-only baselines
14. Build cross-asset context features
15. Build time-of-day and weekday features
16. Create chronological train, validation, and locked test partitions
17. Apply purge or embargo around split boundaries
18. Build XGBoost tabular datasets
19. Build transformer sequences ending only on valid target rows
20. Train XGBoost baseline
21. Train transformer predictor
22. Evaluate classification and calibration
23. Train optional HMM regime detector using training data only
24. Compare models with and without HMM features
25. Generate prediction probabilities
26. Build signal-state features
27. Tune rule-based trading thresholds using validation only
28. Build and train RL trading agents
29. Run transaction-cost-adjusted backtests
30. Compare all model and trading variants
31. Evaluate the locked final test period once the methodology is finalized

---

# Updated Initial Development Plan

## Phase 1: Data Pipeline — Completed

* Downloaded per-symbol Alpaca SIP data
* Converted timestamps to Pacific Time
* Retained 21 pre-open context candles
* Retained 78 regular-session candles
* Removed incomplete days
* Intersected complete days across symbols
* Aligned NVDA, QQQ, and SPY by timestamp

## Phase 2: Triple-Barrier Labeling — Completed

* Used next-candle open as the barrier reference
* Evaluated the next 12 candles
* Assigned neutral, up, or down based on first barrier reached
* Marked same-candle double hits as ambiguous
* Excluded the final 12 regular-session candles from new labels
* Stored bars-to-barrier for up and down events
* Verified each day has 66 labelable targets

## Phase 3: Feature Engineering — Next

Build the first causal feature set:

* Log returns
* Candle body percentage
* Candle range percentage
* Upper and lower wick percentages
* Close position inside candle range
* VWAP distance
* Log volume
* Log trade count
* Average trade size
* Dollar volume
* Volume and trade-count changes
* Time-adjusted relative volume
* QQQ and SPY context features
* NVDA relative returns
* Normalized time of day
* One-hot weekday features

Do not include EMA features.

## Phase 4: Chronological Dataset Splitting

* Lock the final out-of-sample test period
* Create walk-forward development folds
* Add a 12-candle purge or embargo near split boundaries
* Fit all preprocessing statistics on training data only

## Phase 5: XGBoost Baseline

* Build a strong lag-enhanced tabular baseline
* Predict neutral, up, and down probabilities
* Evaluate class performance and calibration
* Tune using validation data only

## Phase 6: Transformer Predictor

* Build fixed-length sequences from per-candle features
* Use pre-open candles as sequence context
* Include only valid target rows in the supervised loss
* Exclude ambiguous rows as targets
* Compare multiple sequence lengths using validation data

## Phase 7: HMM Regime Detection

* Train HMM using training data only
* Generate regime probabilities and regime-state features
* Compare XGBoost and transformer performance with and without regimes

## Phase 8: Rule-Based Trading Baseline

* Build probability-threshold trading rules
* Include realistic costs and slippage
* Tune thresholds using validation data only

## Phase 9: Reinforcement Learning

* Build daily trading episodes
* Use market state, model probabilities, and portfolio state
* Prevent pre-open and overnight trading
* Train RL policies without accessing the final test period

## Phase 10: Final Evaluation

* Compare all major model and trading variants
* Run ablation studies
* Evaluate calibration and prediction quality
* Evaluate transaction-cost-adjusted trading performance
* Run the locked final test period once the methodology is fixed


### imports of major libraries and scripts

In [2]:
#autoreload 
%load_ext autoreload
%autoreload 2
# Core Python
import os
from datetime import datetime, timedelta
from pathlib import Path

# Environment variables
from dotenv import load_dotenv

# Data handling
import numpy as np
import pandas as pd
import polars as pl

# Plotting
import plotly.graph_objects as go
import plotly.express as px

# Progress bars
from tqdm.auto import tqdm

# Alpaca market data
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from alpaca.data.requests import StockLatestTradeRequest

# Project paths
from quant_research.utils.paths import (
    PROJECT_ROOT,
    DATA_DIR,
    RAW_DATA_DIR,
    ALPACA_RAW_DIR,
    PROCESSED_DATA_DIR,
    NOTEBOOKS_DIR,
    RESULTS_DIR,
    LOGS_DIR,
    ensure_project_dirs,
)

# Setup
load_dotenv(PROJECT_ROOT / ".env")
ensure_project_dirs()

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Project root:", PROJECT_ROOT)
print("Raw Alpaca data:", ALPACA_RAW_DIR)
print("Processed data:", PROCESSED_DATA_DIR)

Project root: /Users/rchbeir/Desktop/Quant work/quant_proj
Raw Alpaca data: /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca
Processed data: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed


/Users/rchbeir/Desktop/Quant work/quant_proj/proj/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Now we are testing the Alpaca connection

In [3]:
from quant_research.data.data_downloader import CheckConnection
CheckConnection()

Alpaca connected


### Downloading the actual data

In [4]:
from datetime import datetime
from zoneinfo import ZoneInfo

from quant_research.utils.paths import ALPACA_RAW_DIR
from quant_research.data.data_downloader import Download_5_min_Data

symbols = ["NVDA", "QQQ", "SPY"]
# Alpaca expects timezone-aware datetimes.
# This requests roughly 10 years of historical 5-minute data.
# Format: datetime(year, month, day, hour, minute, tzinfo=ZoneInfo("UTC"))
# 14:30 UTC is 6:30 AM Pacific during standard time, which is the U.S. market open.
# The end date is exclusive, so this covers from Jan 2, 2016 up to Jan 2, 2026.

start = datetime(2016, 1, 2, 14, 30, tzinfo=ZoneInfo("UTC"))
end = datetime(2026, 1, 2, 21, 0, tzinfo=ZoneInfo("UTC"))

df = Download_5_min_Data(
    symbols=symbols,
    start=start,
    end=end,
    timeframe="5min",
    directory=ALPACA_RAW_DIR,
)


NVDA: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/NVDA_5min.parquet
Would you like to overwrite it? (y/n):  n


NVDA: skipping.


QQQ: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/QQQ_5min.parquet
Would you like to overwrite it? (y/n):  n


QQQ: skipping.


SPY: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/SPY_5min.parquet
Would you like to overwrite it? (y/n):  n


SPY: skipping.


### Now we have to check for overlapping days and store them

In [9]:
from quant_research.data.data_processor import save_overlapping_days
save_overlapping_days(symbols)

Transformed NVDA to pacific time
outside candles removed
NVDA has 2515 total days after time filtering
NVDA has 1610 complete days
Transformed QQQ to pacific time
outside candles removed
QQQ has 2515 total days after time filtering
QQQ has 2203 complete days
Transformed SPY to pacific time
outside candles removed
SPY has 2515 total days after time filtering
SPY has 2462 complete days
initial cleaning complete, there are 1539 in common


NVDA: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/NVDA_5min.parquet
Would you like to save processed data it? (y/n):  y


NVDA: proceeding.
Saved processed NVDA: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/NVDA_5min.parquet


QQQ: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/QQQ_5min.parquet
Would you like to save processed data it? (y/n):  y


QQQ: proceeding.
Saved processed QQQ: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/QQQ_5min.parquet


SPY: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/SPY_5min.parquet
Would you like to save processed data it? (y/n):  y


SPY: proceeding.
Saved processed SPY: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/SPY_5min.parquet


In [5]:
#quick sanity check to see if they have all the same number of row
for symbol in symbols:
    path = PROCESSED_DATA_DIR / f"{symbol}_5min.parquet"
    df = pd.read_parquet(path)
    print(symbol)
    print("shape:", df.shape)
    print("unique days:", df["date"].nunique())
    print("start:", df["timestamp"].min())
    print("end:", df["timestamp"].max())
    print()
df.head()

NVDA
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00

QQQ
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00

SPY
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00



,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,timestamp_pt,date,time
0,SPY,2016-02-18 12:45:00+00:00,193.34,193.34,193.19,193.24,89213.0,113.0,193.242909,2016-02-18 04:45:00-08:00,2016-02-18,04:45:00
1,SPY,2016-02-18 12:50:00+00:00,193.24,193.36,193.22,193.35,72778.0,90.0,193.309160,2016-02-18 04:50:00-08:00,2016-02-18,04:50:00
2,SPY,2016-02-18 12:55:00+00:00,193.36,193.46,193.35,193.44,34285.0,67.0,193.426847,2016-02-18 04:55:00-08:00,2016-02-18,04:55:00
3,SPY,2016-02-18 13:00:00+00:00,193.41,193.65,193.22,193.40,86180.0,140.0,193.416101,2016-02-18 05:00:00-08:00,2016-02-18,05:00:00
4,SPY,2016-02-18 13:05:00+00:00,193.40,193.57,193.38,193.45,76086.0,166.0,193.486128,2016-02-18 05:05:00-08:00,2016-02-18,05:05:00


In [6]:
from quant_research.data.data_aligner import build_aligned_dataset

symbols = ["NVDA", "QQQ", "SPY"]

aligned = build_aligned_dataset(
    symbols=symbols,
    timeframe="5min",
)

aligned.head()

Loading and aligning NVDA...
Loading and aligning QQQ...
Loading and aligning SPY...
Saved aligned dataset: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/aligned_5min.parquet


,timestamp,NVDA_open,NVDA_high,NVDA_low,NVDA_close,NVDA_volume,NVDA_trade_count,NVDA_vwap,timestamp_pt,date,time,QQQ_open,QQQ_high,QQQ_low,QQQ_close,QQQ_volume,QQQ_trade_count,QQQ_vwap,SPY_open,SPY_high,SPY_low,SPY_close,SPY_volume,SPY_trade_count,SPY_vwap
0,2016-02-18 12:45:00+00:00,29.82,29.82,29.68,29.73,5504.0,17.0,29.755625,2016-02-18 04:45:00-08:00,2016-02-18,04:45:00,102.87,102.87,102.83,102.83,3212.0,7.0,102.859408,193.34,193.34,193.19,193.24,89213.0,113.0,193.242909
1,2016-02-18 12:50:00+00:00,29.73,29.74,29.62,29.74,4408.0,28.0,29.720000,2016-02-18 04:50:00-08:00,2016-02-18,04:50:00,102.83,102.90,102.83,102.90,3950.0,8.0,102.870633,193.24,193.36,193.22,193.35,72778.0,90.0,193.309160
2,2016-02-18 12:55:00+00:00,29.74,29.74,29.74,29.74,1035.0,3.0,29.740000,2016-02-18 04:55:00-08:00,2016-02-18,04:55:00,102.91,102.91,102.91,102.91,1400.0,3.0,102.910000,193.36,193.46,193.35,193.44,34285.0,67.0,193.426847
3,2016-02-18 13:00:00+00:00,29.70,30.03,29.66,29.88,39864.0,91.0,29.792445,2016-02-18 05:00:00-08:00,2016-02-18,05:00:00,102.95,102.95,102.90,102.95,38699.0,42.0,102.932448,193.41,193.65,193.22,193.40,86180.0,140.0,193.416101
4,2016-02-18 13:05:00+00:00,29.86,29.90,29.80,29.80,5635.0,26.0,29.844374,2016-02-18 05:05:00-08:00,2016-02-18,05:05:00,102.95,103.01,102.94,103.00,15499.0,40.0,102.971399,193.40,193.57,193.38,193.45,76086.0,166.0,193.486128


In [7]:
import importlib
import quant_research.data.label_builder as data_labeler


labeled = data_labeler.label_data(
    timeframe="5min",
    pct=0.01,
    Horizon_bars=12,
)

labeled.head()

KeyboardInterrupt: 

In [ ]:
import importlib
import quant_research.features.feature_builder as feature_builder

importlib.reload(feature_builder)

feature_builder.build_features()
feature_builder.build_model_feature_dataset()

### Creating the train validate test sets

In [8]:
import importlib
import quant_research.features.feature_builder as feature_builder

importlib.reload(feature_builder)

walk_forward_manifest = feature_builder.build_walk_forward_splits(
    timeframe="5min",
    test_fraction=0.20,
    initial_train_days=504,
    validation_days=63,
    step_days=63,
)

display(walk_forward_manifest)

Locked test: 308 days, 30,492 total rows
Test dates: 2024-10-07 through 2026-01-02

Fold 1
Train: 2016-02-18 through 2021-09-01 (504 days)
Validation: 2021-09-02 through 2021-12-17 (63 days)
Valid targets: 33,249 train, 4,156 validation

Fold 2
Train: 2016-02-18 through 2021-12-17 (567 days)
Validation: 2021-12-20 through 2022-03-23 (63 days)
Valid targets: 37,405 train, 4,157 validation

Fold 3
Train: 2016-02-18 through 2022-03-23 (630 days)
Validation: 2022-03-24 through 2022-06-28 (63 days)
Valid targets: 41,562 train, 4,148 validation

Fold 4
Train: 2016-02-18 through 2022-06-28 (693 days)
Validation: 2022-06-30 through 2022-10-11 (63 days)
Valid targets: 45,710 train, 4,158 validation

Fold 5
Train: 2016-02-18 through 2022-10-11 (756 days)
Validation: 2022-10-13 through 2023-01-26 (63 days)
Valid targets: 49,868 train, 4,157 validation

Fold 6
Train: 2016-02-18 through 2023-01-26 (819 days)
Validation: 2023-01-27 through 2023-05-01 (63 days)
Valid targets: 54,025 train, 4,158 vali

,fold,train_start,train_end,validation_start,validation_end,train_days,validation_days,train_rows,validation_rows,train_valid_targets,validation_valid_targets
0,1,2016-02-18,2021-09-01,2021-09-02,2021-12-17,504,63,49896,6237,33249,4156
1,2,2016-02-18,2021-12-17,2021-12-20,2022-03-23,567,63,56133,6237,37405,4157
2,3,2016-02-18,2022-03-23,2022-03-24,2022-06-28,630,63,62370,6237,41562,4148
3,4,2016-02-18,2022-06-28,2022-06-30,2022-10-11,693,63,68607,6237,45710,4158
4,5,2016-02-18,2022-10-11,2022-10-13,2023-01-26,756,63,74844,6237,49868,4157
5,6,2016-02-18,2023-01-26,2023-01-27,2023-05-01,819,63,81081,6237,54025,4158
6,7,2016-02-18,2023-05-01,2023-05-02,2023-08-04,882,63,87318,6237,58183,4158
7,8,2016-02-18,2023-08-04,2023-08-07,2023-11-02,945,63,93555,6237,62341,4158
8,9,2016-02-18,2023-11-02,2023-11-03,2024-02-12,1008,63,99792,6237,66499,4158
9,10,2016-02-18,2024-02-12,2024-02-13,2024-05-13,1071,63,106029,6237,70657,4158


## Run XGboost initial training

In [21]:
import importlib
import quant_research.models.xgboost_walk_forward as xgb_walk_forward

importlib.reload(xgb_walk_forward)

xgboost_summary_lagged = xgb_walk_forward.train_xgboost_walk_forward(
    timeframe="5min",
    early_stopping_fraction=0.10,
    use_balanced_sample_weights=False,
    random_seed=42,
)

display(xgboost_summary_lagged)

Training XGBoost walk-forward folds:   0%|             | 0/11 [00:00<?, ?fold/s]

Training fold_01:   0%|                                | 0/11 [00:00<?, ?fold/s]
Training fold_01:   0%| | 0/11 [00:02<?, ?fold/s, macro_f1=0.4677, accuracy=0.66
Training fold_01:   9%| | 1/11 [00:02<00:22,  2.30s/fold, macro_f1=0.4677, accur
Training fold_02:   9%| | 1/11 [00:02<00:22,  2.30s/fold, macro_f1=0.4677, accur


fold_01
Fit dates: 2016-02-18 through 2021-03-17
Validation dates: 2021-09-02 through 2021-12-17
Best iteration: 169
Accuracy: 0.6675
Balanced accuracy: 0.4654
Macro F1: 0.4677
Log loss: 0.7244



Training fold_02:   9%| | 1/11 [00:05<00:22,  2.30s/fold, macro_f1=0.4556, accur
Training fold_02:  18%|▏| 2/11 [00:05<00:23,  2.59s/fold, macro_f1=0.4556, accur
Training fold_03:  18%|▏| 2/11 [00:05<00:23,  2.59s/fold, macro_f1=0.4556, accur


fold_02
Fit dates: 2016-02-18 through 2021-09-13
Validation dates: 2021-12-20 through 2022-03-23
Best iteration: 220
Accuracy: 0.4958
Balanced accuracy: 0.4735
Macro F1: 0.4556
Log loss: 0.9518



Training fold_03:  18%|▏| 2/11 [00:07<00:23,  2.59s/fold, macro_f1=0.4607, accur
Training fold_03:  27%|▎| 3/11 [00:07<00:20,  2.58s/fold, macro_f1=0.4607, accur
Training fold_04:  27%|▎| 3/11 [00:07<00:20,  2.58s/fold, macro_f1=0.4607, accur

...


Training fold_10:  82%|▊| 9/11 [00:27<00:05,  2.95s/fold, macro_f1=0.4363, accur
Training fold_10:  91%|▉| 10/11 [00:27<00:02,  2.96s/fold, macro_f1=0.4363, accu
Training fold_11:  91%|▉| 10/11 [00:27<00:02,  2.96s/fold, macro_f1=0.4363, accu


fold_10
Fit dates: 2016-02-18 through 2023-08-31
Validation dates: 2024-02-13 through 2024-05-13
Best iteration: 167
Accuracy: 0.6657
Balanced accuracy: 0.4370
Macro F1: 0.4363
Log loss: 0.8038



Training fold_11:  91%|▉| 10/11 [00:30<00:02,  2.96s/fold, macro_f1=0.4455, accu
Training fold_11: 100%|█| 11/11 [00:30<00:00,  2.78s/fold, macro_f1=0.4455, accu


fold_11
Fit dates: 2016-02-18 through 2023-11-21
Validation dates: 2024-05-15 through 2024-08-16
Best iteration: 130
Accuracy: 0.5999
Balanced accuracy: 0.4590
Macro F1: 0.4455
Log loss: 0.8660

Walk-forward fold results
   fold  accuracy  balanced_accuracy  macro_f1  weighted_f1  log_loss  multiclass_brier_score  best_iteration
fold_01  0.667469           0.465430  0.467686     0.625457  0.724378                0.414576             169
fold_02  0.495790           0.473511  0.455625     0.469868  0.951763                0.575686             220
fold_03  0.476374           0.479962  0.460747     0.458733  0.984619                0.600224             183
fold_04  0.580327           0.461222  0.460880     0.543590  0.925223                0.544901             213
fold_05  0.570363           0.457397  0.450424     0.530041  0.940870                0.553199              96
fold_06  0.678932           0.429467  0.430495     0.625110  0.775238                0.438831             201
fold_07 

,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score,fold,fit_start,fit_end,early_stopping_start,early_stopping_end,validation_start,validation_end,fit_rows,early_stopping_rows,validation_rows,best_iteration,best_score,balanced_sample_weights
0,0.667469,0.465430,0.467686,0.625457,0.724378,0.414576,fold_01,2016-02-18,2021-03-17,2021-04-13,2021-09-01,2021-09-02,2021-12-17,29949,3300,4156,169,0.602365,False
1,0.495790,0.473511,0.455625,0.469868,0.951763,0.575686,fold_02,2016-02-18,2021-09-13,2021-09-14,2021-12-17,2021-12-20,2022-03-23,33711,3694,4157,220,0.768598,False
2,0.476374,0.479962,0.460747,0.458733,0.984619,0.600224,fold_03,2016-02-18,2021-12-17,2021-12-20,2022-03-23,2022-03-24,2022-06-28,37405,4157,4148,183,0.946267,False
3,0.580327,0.461222,0.460880,0.543590,0.925223,0.544901,fold_04,2016-02-18,2022-03-15,2022-03-16,2022-06-28,2022-06-30,2022-10-11,41166,4544,4158,213,0.987787,False
4,0.570363,0.457397,0.450424,0.530041,0.940870,0.553199,fold_05,2016-02-18,2022-06-08,2022-06-09,2022-10-11,2022-10-13,2023-01-26,44926,4942,4157,96,0.931219,False
5,0.678932,0.429467,0.430495,0.625110,0.775238,0.438831,fold_06,2016-02-18,2022-09-13,2022-09-14,2023-01-26,2023-01-27,2023-05-01,48680,5345,4158,201,0.948935,False
6,0.723665,0.384905,0.377172,0.645073,0.708486,0.394089,fold_07,2016-02-18,2022-12-13,2022-12-14,2023-05-01,2023-05-02,2023-08-04,52375,5808,4158,145,0.810992,False
7,0.727754,0.421113,0.428217,0.669188,0.726459,0.403195,fold_08,2016-02-18,2023-03-14,2023-03-15,2023-08-04,2023-08-07,2023-11-02,56137,6204,4158,284,0.687931,False
8,0.809764,0.372647,0.374545,0.749272,0.549288,0.290048,fold_09,2016-02-18,2023-06-07,2023-06-09,2023-11-02,2023-11-03,2024-02-12,59899,6600,4158,142,0.714657,False
9,0.665705,0.436987,0.436254,0.606367,0.803786,0.459146,fold_10,2016-02-18,2023-08-31,2023-09-01,2024-02-12,2024-02-13,2024-05-13,63595,7062,4158,167,0.595000,False


In [22]:
from pathlib import Path
import json
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

results_dir = Path(PROCESSED_DATA_DIR) / "xgboost_walk_forward_5min"

rows = []

for report_path in sorted(results_dir.glob("fold_*/classification_report.json")):
    fold = report_path.parent.name

    with report_path.open("r") as file:
        report = json.load(file)

    for class_name in ["neutral", "up", "down"]:
        rows.append({
            "fold": fold,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1": report[class_name]["f1-score"],
            "support": report[class_name]["support"],
        })

per_class = pd.DataFrame(rows)

display(per_class)

print("\nMean per-class metrics:")
display(
    per_class
    .groupby("class")[["precision", "recall", "f1"]]
    .mean()
    .sort_index()
)

,fold,class,precision,recall,f1,support
0,fold_01,neutral,0.743574,0.919219,0.822120,2612.0
1,fold_01,up,0.429104,0.154570,0.227273,744.0
2,fold_01,down,0.391502,0.322500,0.353667,800.0
3,fold_02,neutral,0.569238,0.800254,0.665261,1577.0
4,fold_02,up,0.438356,0.374707,0.404040,1281.0
5,fold_02,down,0.377515,0.245574,0.297575,1299.0
6,fold_03,neutral,0.518200,0.758697,0.615801,1351.0
7,fold_03,up,0.437500,0.349194,0.388391,1303.0
8,fold_03,down,0.438938,0.331995,0.378049,1494.0
9,fold_04,neutral,0.645325,0.859657,0.737229,2216.0



Mean per-class metrics:


,precision,recall,f1
class,,,
down,0.383378,0.210665,0.262528
neutral,0.690335,0.904154,0.782301
up,0.423927,0.205620,0.260857


In [23]:
rows = []

for pred_path in sorted(results_dir.glob("fold_*/validation_predictions.parquet")):
    fold = pred_path.parent.name
    pred = pd.read_parquet(pred_path)

    actual_counts = pred["target_name"].value_counts(normalize=True)
    predicted_counts = pred["predicted_name"].value_counts(normalize=True)

    for class_name in ["neutral", "up", "down"]:
        rows.append({
            "fold": fold,
            "class": class_name,
            "actual_pct": actual_counts.get(class_name, 0.0),
            "predicted_pct": predicted_counts.get(class_name, 0.0),
        })

dist = pd.DataFrame(rows)

print("\nAverage actual vs predicted distribution:")
display(
    dist
    .groupby("class")[["actual_pct", "predicted_pct"]]
    .mean()
    .sort_index()
)


Average actual vs predicted distribution:


,actual_pct,predicted_pct
class,,
down,0.210941,0.123259
neutral,0.586766,0.764048
up,0.202293,0.112693


In [25]:
import importlib
import quant_research.models.xgboost_walk_forward as xgb_walk_forward

importlib.reload(xgb_walk_forward)

xgboost_summary_lagged = xgb_walk_forward.train_xgboost_walk_forward(
    timeframe="5min",
    early_stopping_fraction=0.10,
    use_balanced_sample_weights=True,
    random_seed=42,
)

display(xgboost_summary_lagged)

Training XGBoost walk-forward folds:   0%|             | 0/11 [00:00<?, ?fold/s]

Training fold_01:   0%|                                | 0/11 [00:00<?, ?fold/s]
Training fold_01:   0%| | 0/11 [00:02<?, ?fold/s, macro_f1=0.4958, accuracy=0.58
Training fold_01:   9%| | 1/11 [00:02<00:22,  2.24s/fold, macro_f1=0.4958, accur
Training fold_02:   9%| | 1/11 [00:02<00:22,  2.24s/fold, macro_f1=0.4958, accur


fold_01
Fit dates: 2016-02-18 through 2021-03-17
Validation dates: 2021-09-02 through 2021-12-17
Best iteration: 156
Accuracy: 0.5808
Balanced accuracy: 0.5159
Macro F1: 0.4958
Log loss: 0.8974



Training fold_02:   9%| | 1/11 [00:16<00:22,  2.24s/fold, macro_f1=0.4328, accur
Training fold_02:  18%|▏| 2/11 [00:16<01:21,  9.08s/fold, macro_f1=0.4328, accur
Training fold_03:  18%|▏| 2/11 [00:16<01:21,  9.08s/fold, macro_f1=0.4328, accur


fold_02
Fit dates: 2016-02-18 through 2021-09-13
Validation dates: 2021-12-20 through 2022-03-23
Best iteration: 1478
Accuracy: 0.4325
Balanced accuracy: 0.4352
Macro F1: 0.4328
Log loss: 1.0729



Training fold_03:  18%|▏| 2/11 [00:17<01:21,  9.08s/fold, macro_f1=0.4172, accur
Training fold_03:  27%|▎| 3/11 [00:17<00:44,  5.55s/fold, macro_f1=0.4172, accur
Training fold_04:  27%|▎| 3/11 [00:17<00:44,  5.55s/fold, macro_f1=0.4172, accur

...


Training fold_10:  82%|▊| 9/11 [01:22<00:22, 11.12s/fold, macro_f1=0.4185, accur
Training fold_10:  91%|▉| 10/11 [01:22<00:14, 14.23s/fold, macro_f1=0.4185, accu
Training fold_11:  91%|▉| 10/11 [01:22<00:14, 14.23s/fold, macro_f1=0.4185, accu


fold_10
Fit dates: 2016-02-18 through 2023-08-31
Validation dates: 2024-02-13 through 2024-05-13
Best iteration: 1996
Accuracy: 0.5334
Balanced accuracy: 0.4325
Macro F1: 0.4185
Log loss: 0.9629



Training fold_11:  91%|▉| 10/11 [01:26<00:14, 14.23s/fold, macro_f1=0.4285, accu
Training fold_11: 100%|█| 11/11 [01:26<00:00,  7.87s/fold, macro_f1=0.4285, accu


fold_11
Fit dates: 2016-02-18 through 2023-11-21
Validation dates: 2024-05-15 through 2024-08-16
Best iteration: 313
Accuracy: 0.4714
Balanced accuracy: 0.4393
Macro F1: 0.4285
Log loss: 0.9983

Walk-forward fold results
   fold  accuracy  balanced_accuracy  macro_f1  weighted_f1  log_loss  multiclass_brier_score  best_iteration
fold_01  0.580847           0.515917  0.495834     0.614047  0.897414                0.522584             156
fold_02  0.432523           0.435214  0.432783     0.438946  1.072896                0.652165            1478
fold_03  0.416104           0.417432  0.417183     0.415973  1.041890                0.634068              50
fold_04  0.462241           0.440137  0.429835     0.480990  1.043902                0.625906              53
fold_05  0.438537           0.413384  0.406659     0.456633  1.055783                0.636910              41
fold_06  0.578403           0.457479  0.449799     0.602692  0.939763                0.551462              53
fold_07 

,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score,fold,fit_start,fit_end,early_stopping_start,early_stopping_end,validation_start,validation_end,fit_rows,early_stopping_rows,validation_rows,best_iteration,best_score,balanced_sample_weights
0,0.580847,0.515917,0.495834,0.614047,0.897414,0.522584,fold_01,2016-02-18,2021-03-17,2021-04-13,2021-09-01,2021-09-02,2021-12-17,29949,3300,4156,156,0.777947,True
1,0.432523,0.435214,0.432783,0.438946,1.072896,0.652165,fold_02,2016-02-18,2021-09-13,2021-09-14,2021-12-17,2021-12-20,2022-03-23,33711,3694,4157,1478,0.845239,True
2,0.416104,0.417432,0.417183,0.415973,1.041890,0.634068,fold_03,2016-02-18,2021-12-17,2021-12-20,2022-03-23,2022-03-24,2022-06-28,37405,4157,4148,50,1.032954,True
3,0.462241,0.440137,0.429835,0.480990,1.043902,0.625906,fold_04,2016-02-18,2022-03-15,2022-03-16,2022-06-28,2022-06-30,2022-10-11,41166,4544,4158,53,1.041298,True
4,0.438537,0.413384,0.406659,0.456633,1.055783,0.636910,fold_05,2016-02-18,2022-06-08,2022-06-09,2022-10-11,2022-10-13,2023-01-26,44926,4942,4157,41,1.046370,True
5,0.578403,0.457479,0.449799,0.602692,0.939763,0.551462,fold_06,2016-02-18,2022-09-13,2022-09-14,2023-01-26,2023-01-27,2023-05-01,48680,5345,4158,53,1.054332,True
6,0.657528,0.455359,0.454078,0.659253,0.845056,0.485156,fold_07,2016-02-18,2022-12-13,2022-12-14,2023-05-01,2023-05-02,2023-08-04,52375,5808,4158,111,0.952307,True
7,0.643338,0.469909,0.463162,0.654398,0.869061,0.498745,fold_08,2016-02-18,2023-03-14,2023-03-15,2023-08-04,2023-08-07,2023-11-02,56137,6204,4158,1486,0.789810,True
8,0.753968,0.433972,0.435883,0.750712,0.647352,0.345778,fold_09,2016-02-18,2023-06-07,2023-06-09,2023-11-02,2023-11-03,2024-02-12,59899,6600,4158,1981,0.820712,True
9,0.533430,0.432515,0.418506,0.556426,0.962915,0.559897,fold_10,2016-02-18,2023-08-31,2023-09-01,2024-02-12,2024-02-13,2024-05-13,63595,7062,4158,1996,0.719964,True


In [26]:
from pathlib import Path
import json
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

results_dir = Path(PROCESSED_DATA_DIR) / "xgboost_walk_forward_5min_balanced"

rows = []

for report_path in sorted(results_dir.glob("fold_*/classification_report.json")):
    fold = report_path.parent.name

    with report_path.open("r") as file:
        report = json.load(file)

    for class_name in ["neutral", "up", "down"]:
        rows.append({
            "fold": fold,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1": report[class_name]["f1-score"],
            "support": report[class_name]["support"],
        })

per_class_balanced = pd.DataFrame(rows)

display(per_class_balanced)

print("\nMean per-class metrics:")
display(
    per_class_balanced
    .groupby("class")[["precision", "recall", "f1"]]
    .mean()
    .sort_index()
)

,fold,class,precision,recall,f1,support
0,fold_01,neutral,0.899325,0.663476,0.763604,2612.0
1,fold_01,up,0.328037,0.471774,0.386990,744.0
2,fold_01,down,0.284728,0.412500,0.336907,800.0
3,fold_02,neutral,0.753247,0.404566,0.526403,1577.0
4,fold_02,up,0.373500,0.583138,0.455349,1281.0
5,fold_02,down,0.315267,0.317937,0.316596,1299.0
6,fold_03,neutral,0.657321,0.312361,0.423482,1351.0
7,fold_03,up,0.376860,0.524942,0.438743,1303.0
8,fold_03,down,0.366647,0.414993,0.389325,1494.0
9,fold_04,neutral,0.722861,0.510830,0.598625,2216.0



Mean per-class metrics:


,precision,recall,f1
class,,,
down,0.268964,0.319113,0.289483
neutral,0.789924,0.615368,0.683973
up,0.308928,0.404774,0.344412


In [27]:
rows = []

for pred_path in sorted(results_dir.glob("fold_*/validation_predictions.parquet")):
    fold = pred_path.parent.name
    pred = pd.read_parquet(pred_path)

    actual_counts = pred["target_name"].value_counts(normalize=True)
    predicted_counts = pred["predicted_name"].value_counts(normalize=True)

    for class_name in ["neutral", "up", "down"]:
        rows.append({
            "fold": fold,
            "class": class_name,
            "actual_pct": actual_counts.get(class_name, 0.0),
            "predicted_pct": predicted_counts.get(class_name, 0.0),
        })

dist_balanced = pd.DataFrame(rows)

print("\nAverage actual vs predicted distribution:")
display(
    dist_balanced
    .groupby("class")[["actual_pct", "predicted_pct"]]
    .mean()
    .sort_index()
)


Average actual vs predicted distribution:


,actual_pct,predicted_pct
class,,
down,0.210941,0.248606
neutral,0.586766,0.473781
up,0.202293,0.277614


## XGBoost Walk-Forward Results

We evaluated two XGBoost baselines using expanding walk-forward validation:

1. Lagged unweighted XGBoost
2. Lagged balanced XGBoost

Both models used the same engineered feature set, including current-candle price/volume features, QQQ/SPY context features, time features, and lagged/rolling history features. The difference is that the balanced model used class-balanced sample weights during training, while the unweighted model trained on the natural class distribution.

---

### Lagged Unweighted XGBoost

The lagged unweighted model achieved:

| Metric | Mean Value |
|---|---:|
| Accuracy | 0.6360 |
| Balanced Accuracy | 0.4401 |
| Macro F1 | 0.4352 |
| Weighted F1 | 0.5882 |
| Log Loss | 0.8142 |
| Multiclass Brier Score | 0.4707 |

This model had the best probability quality, shown by the lower log loss and lower Brier score. However, it still leaned heavily toward predicting the majority class, neutral.

The average actual validation distribution was:

| Class | Actual % |
|---|---:|
| Neutral | 58.7% |
| Up | 20.2% |
| Down | 21.1% |

The unweighted model predicted:

| Class | Predicted % |
|---|---:|
| Neutral | 76.4% |
| Up | 11.3% |
| Down | 12.3% |

This means the model was conservative and overpredicted neutral. It performed well on neutral examples but had weaker recall for directional moves.

Mean per-class F1:

| Class | F1 |
|---|---:|
| Neutral | 0.7823 |
| Up | 0.2609 |
| Down | 0.2625 |

---

### Lagged Balanced XGBoost

The lagged balanced model achieved:

| Metric | Mean Value |
|---|---:|
| Accuracy | 0.5426 |
| Balanced Accuracy | 0.4464 |
| Macro F1 | 0.4393 |
| Weighted F1 | 0.5573 |
| Log Loss | 0.9431 |
| Multiclass Brier Score | 0.5556 |

The balanced model slightly improved macro F1 and balanced accuracy, meaning it did a better job treating all three classes more evenly. However, it produced worse probability quality, with higher log loss and a higher Brier score.

The balanced model predicted:

| Class | Predicted % |
|---|---:|
| Neutral | 47.4% |
| Up | 27.8% |
| Down | 24.9% |

This means class weighting shifted the model away from overpredicting neutral and made it predict more up and down events. However, it overcorrected and predicted directional moves more often than they actually occurred.

Mean per-class F1:

| Class | F1 |
|---|---:|
| Neutral | 0.6840 |
| Up | 0.3444 |
| Down | 0.2895 |

Compared with the unweighted model, the balanced model improved directional F1:

| Class | Unweighted F1 | Balanced F1 |
|---|---:|---:|
| Up | 0.2609 | 0.3444 |
| Down | 0.2625 | 0.2895 |

But it hurt neutral classification:

| Class | Unweighted F1 | Balanced F1 |
|---|---:|---:|
| Neutral | 0.7823 | 0.6840 |

---

### Main Interpretation

The two models show a clear tradeoff.

The unweighted model is better calibrated and produces better probability estimates. It is more conservative and tends to predict neutral too often. This makes it better suited for probability-threshold trading experiments, where we only act when the model is highly confident.

The balanced model improves directional recall and directional F1, especially for the up class. However, it does this by predicting more up and down events overall, which damages probability calibration and increases log loss.

Therefore, the current best baselines are:

- Best probability baseline: lagged unweighted XGBoost
- Best directional classification baseline: lagged balanced XGBoost

---

### Conclusion

The XGBoost models are learning real signal, but the directional prediction problem remains difficult. The unweighted model is too conservative, while the balanced model is more aggressive but less calibrated.

The next step is not simply to pick the highest-accuracy model. Instead, we should evaluate whether the model probabilities can generate useful trading signals.

The next experiment should be a probability-threshold sweep using rules such as:

text Long if P(up) - P(down) >= threshold Short if P(down) - P(up) >= threshold Flat otherwise 

This will show whether the predicted probabilities are useful for selecting high-confidence long and short trades, even if raw argmax classification is imperfect.

## Now we run a sweep to see the baselines

In [9]:
import importlib
import quant_research.evaluation.threshold_sweep as threshold_sweep

importlib.reload(threshold_sweep)

threshold_outputs = threshold_sweep.run_default_xgboost_threshold_sweeps()

Experiment: lagged_unweighted
Loaded predictions: 45,722 rows
Saved summary to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/threshold_sweep_5min/lagged_unweighted/threshold_sweep_summary.csv
Saved fold results to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/threshold_sweep_5min/lagged_unweighted/threshold_sweep_by_fold.csv

Top settings by directional precision:
 edge_threshold  min_direction_probability  signal_rate  long_precision  short_precision  directional_precision  directional_edge_over_base
          0.300                       0.50     0.002209        0.558140         0.482759               0.514851                    0.101681
          0.300                       0.45     0.002362        0.500000         0.466667               0.481481                    0.068311
          0.300                        NaN     0.002428        0.470588         0.466667               0.468468                    0.055298
          0.300                       0.30 

## Threshold Sweep Interpretation

After training the XGBoost models, we tested whether the predicted probabilities could be used to create higher-confidence trading signals.

Instead of always choosing the class with the highest probability, we applied a threshold rule:

text Long if P(up) - P(down) >= threshold Short if P(down) - P(up) >= threshold Flat otherwise 

We also tested a minimum directional probability requirement, such as:

text P(up) >= 0.50 for long signals P(down) >= 0.50 for short signals 

The purpose of this experiment was to answer:

When the model is confident enough to give a directional signal, is it right more often than the base rate?

---

### Lagged Unweighted XGBoost

The lagged unweighted model produced the most useful probability-based signals.

A representative setting was:

| Parameter | Value |
|---|---:|
| Edge Threshold | 0.05 |
| Minimum Direction Probability | 0.50 |
| Signal Rate | 4.55% |
| Long Precision | 46.88% |
| Short Precision | 45.23% |
| Directional Precision | 46.25% |
| Directional Edge Over Base | +4.93 percentage points |

This means the model only produced directional signals on about 4.55% of validation rows. Among those long and short signals, it chose the correct direction about 46.25% of the time.

The best very-high-confidence setting reached about 51.49% directional precision, but the signal rate was only 0.22%, meaning there were very few trades. That setting is probably too sparse to rely on.

The more realistic threshold settings showed a directional precision around 45% to 46%, which is better than the base directional rate but still far from a finished trading strategy.

---

### Lagged Balanced XGBoost

The balanced model performed worse in the threshold sweep.

Although class weighting improved the model's directional recall during classification, it damaged the usefulness of its predicted probabilities.

The balanced model's top threshold settings had negative directional edge over the base rate. For example, its strongest displayed setting had:

| Metric | Value |
|---|---:|
| Directional Precision | 37.17% |
| Directional Edge Over Base | -4.15 percentage points |

This means that even when the balanced model gave directional signals, those signals were worse than the base directional rate.

Therefore, the balanced model is not the preferred model for trading-signal thresholding.

---

### Why Directional Precision Below 50% Can Still Matter

The unweighted model's best realistic directional precision was still below 50%, meaning it was wrong more than half the time.

That is important and should not be ignored.

However, trading profitability does not depend only on win rate. A strategy can be profitable with a win rate below 50% if the average winning trade is larger than the average losing trade.

For example:

text Win rate = 46% Average win = +1.2% Average loss = -0.8%  Expected value = 0.46 × 1.2% - 0.54 × 0.8% = +0.12% 

But if wins and losses are both about 1%, then a 46% win rate would likely lose money before costs:

text 0.46 × 1.0% - 0.54 × 1.0% = -0.08% 

After slippage and fees, that would be worse.

So the threshold sweep shows that the model has some directional signal, but it does not prove the signal is tradable yet.

---

### Main Takeaway

The threshold sweep showed:

1. The lagged unweighted XGBoost model has better probability quality than the balanced model.
2. The unweighted model can identify a small subset of higher-confidence directional opportunities.
3. The directional edge is positive but modest.
4. The balanced model is less useful for trading thresholds because its probabilities are poorly calibrated.
5. The model is still wrong more than half the time on realistic directional signals, so this is not yet a trading strategy.

The best current model for the next stage is:

text Lagged unweighted XGBoost 

The next step is to run a simple backtest to measure:

- Average win
- Average loss
- Trade count
- Win rate
- Net return after costs
- Drawdown
- Whether the model's probability edge survives execution timing
- Whether winners are large enough to compensate for being wrong more than half the time

Until the backtest is completed, the result should be interpreted as evidence of a modest predictive edge, not evidence of profitability.


In [71]:
from pathlib import Path
import json
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

processed_dir = Path(PROCESSED_DATA_DIR)

xgb_unweighted_dir = processed_dir / "xgboost_walk_forward_5min"
xgb_balanced_dir = processed_dir / "xgboost_walk_forward_5min_balanced"

unweighted_summary = pd.read_csv(
    xgb_unweighted_dir / "walk_forward_summary.csv"
)

balanced_summary = pd.read_csv(
    xgb_balanced_dir / "walk_forward_summary.csv"
)

model_comparison = pd.DataFrame([
    {
        "model": "lagged_unweighted_xgboost",
        "mean_accuracy": unweighted_summary["accuracy"].mean(),
        "mean_balanced_accuracy": unweighted_summary["balanced_accuracy"].mean(),
        "mean_macro_f1": unweighted_summary["macro_f1"].mean(),
        "mean_weighted_f1": unweighted_summary["weighted_f1"].mean(),
        "mean_log_loss": unweighted_summary["log_loss"].mean(),
        "mean_brier": unweighted_summary["multiclass_brier_score"].mean(),
    },
    {
        "model": "lagged_balanced_xgboost",
        "mean_accuracy": balanced_summary["accuracy"].mean(),
        "mean_balanced_accuracy": balanced_summary["balanced_accuracy"].mean(),
        "mean_macro_f1": balanced_summary["macro_f1"].mean(),
        "mean_weighted_f1": balanced_summary["weighted_f1"].mean(),
        "mean_log_loss": balanced_summary["log_loss"].mean(),
        "mean_brier": balanced_summary["multiclass_brier_score"].mean(),
    },
])

display(model_comparison)

display(
    unweighted_summary[
        [
            "fold",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "log_loss",
            "multiclass_brier_score",
            "best_iteration",
        ]
    ]
)

,model,mean_accuracy,mean_balanced_accuracy,mean_macro_f1,mean_weighted_f1,mean_log_loss,mean_brier
0,lagged_unweighted_xgboost,0.636000,0.440147,0.435229,0.588161,0.814191,0.470712
1,lagged_balanced_xgboost,0.542572,0.446418,0.439289,0.557281,0.943122,0.555592


,fold,accuracy,balanced_accuracy,macro_f1,log_loss,multiclass_brier_score,best_iteration
0,fold_01,0.667469,0.465430,0.467686,0.724378,0.414576,169
1,fold_02,0.495790,0.473511,0.455625,0.951763,0.575686,220
2,fold_03,0.476374,0.479962,0.460747,0.984619,0.600224,183
3,fold_04,0.580327,0.461222,0.460880,0.925223,0.544901,213
4,fold_05,0.570363,0.457397,0.450424,0.940870,0.553199,96
5,fold_06,0.678932,0.429467,0.430495,0.775238,0.438831,201
6,fold_07,0.723665,0.384905,0.377172,0.708486,0.394089,145
7,fold_08,0.727754,0.421113,0.428217,0.726459,0.403195,284
8,fold_09,0.809764,0.372647,0.374545,0.549288,0.290048,142
9,fold_10,0.665705,0.436987,0.436254,0.803786,0.459146,167


In [1]:
def load_per_class_reports(results_dir):
    rows = []

    for report_path in sorted(results_dir.glob("fold_*/classification_report.json")):
        fold = report_path.parent.name

        with report_path.open("r") as file:
            report = json.load(file)

        for class_name in ["neutral", "up", "down"]:
            rows.append({
                "fold": fold,
                "class": class_name,
                "precision": report[class_name]["precision"],
                "recall": report[class_name]["recall"],
                "f1": report[class_name]["f1-score"],
                "support": report[class_name]["support"],
            })

    return pd.DataFrame(rows)


per_class_unweighted = load_per_class_reports(xgb_unweighted_dir)
per_class_balanced = load_per_class_reports(xgb_balanced_dir)

print("Lagged unweighted XGBoost — mean per-class metrics")
display(
    per_class_unweighted
    .groupby("class")[["precision", "recall", "f1"]]
    .mean()
    .sort_index()
)

print("Lagged balanced XGBoost — mean per-class metrics")
display(
    per_class_balanced
    .groupby("class")[["precision", "recall", "f1"]]
    .mean()
    .sort_index()
)

NameError: name 'xgb_unweighted_dir' is not defined

In [73]:
threshold_unweighted_path = (
    processed_dir
    / "threshold_sweep_5min"
    / "lagged_unweighted"
    / "threshold_sweep_summary.csv"
)

threshold_balanced_path = (
    processed_dir
    / "threshold_sweep_5min"
    / "lagged_balanced"
    / "threshold_sweep_summary.csv"
)

threshold_unweighted = pd.read_csv(threshold_unweighted_path)
threshold_balanced = pd.read_csv(threshold_balanced_path)

print("Top lagged unweighted threshold settings by directional precision")
display(
    threshold_unweighted
    .sort_values(
        ["directional_precision", "signal_rate"],
        ascending=[False, False],
    )
    [
        [
            "edge_threshold",
            "min_direction_probability",
            "signal_rate",
            "long_precision",
            "short_precision",
            "directional_precision",
            "directional_edge_over_base",
        ]
    ]
    .head(10)
)

print("Top lagged balanced threshold settings by directional precision")
display(
    threshold_balanced
    .sort_values(
        ["directional_precision", "signal_rate"],
        ascending=[False, False],
    )
    [
        [
            "edge_threshold",
            "min_direction_probability",
            "signal_rate",
            "long_precision",
            "short_precision",
            "directional_precision",
            "directional_edge_over_base",
        ]
    ]
    .head(10)
)

Top lagged unweighted threshold settings by directional precision


,edge_threshold,min_direction_probability,signal_rate,long_precision,short_precision,directional_precision,directional_edge_over_base
65,0.300,0.50,0.002209,0.558140,0.482759,0.514851,0.101681
54,0.300,0.45,0.002362,0.500000,0.466667,0.481481,0.068311
10,0.300,NaN,0.002428,0.470588,0.466667,0.468468,0.055298
21,0.300,0.30,0.002428,0.470588,0.466667,0.468468,0.055298
32,0.300,0.35,0.002428,0.470588,0.466667,0.468468,0.055298
43,0.300,0.40,0.002428,0.470588,0.466667,0.468468,0.055298
57,0.050,0.50,0.045492,0.468847,0.452261,0.462500,0.049329
61,0.150,0.50,0.025655,0.460719,0.459716,0.460358,0.047187
55,0.000,0.50,0.046433,0.466206,0.450670,0.460198,0.047027
56,0.025,0.50,0.046433,0.466206,0.450670,0.460198,0.047027


Top lagged balanced threshold settings by directional precision


,edge_threshold,min_direction_probability,signal_rate,long_precision,short_precision,directional_precision,directional_edge_over_base
55,0.000,0.50,0.116683,0.385059,0.350147,0.371696,-0.041475
56,0.025,0.50,0.116115,0.384451,0.347955,0.370503,-0.042668
57,0.050,0.50,0.114234,0.382762,0.343404,0.367796,-0.045375
58,0.075,0.50,0.111237,0.381508,0.340115,0.365906,-0.047264
65,0.300,0.50,0.043633,0.377489,0.341074,0.364912,-0.048259
44,0.000,0.45,0.218997,0.383307,0.332540,0.364127,-0.049044
59,0.100,0.50,0.106316,0.379763,0.335718,0.363300,-0.049871
60,0.125,0.50,0.100039,0.380638,0.333333,0.363139,-0.050031
45,0.025,0.45,0.212436,0.381941,0.330132,0.362504,-0.050667
61,0.150,0.50,0.091225,0.372901,0.332689,0.357948,-0.055223


In [74]:
# Main timeout-only candidate:
# lagged unweighted XGBoost
# long_edge = 0.05
# short_edge = 0.15
# no TP, no SL, 12-bar timeout exit

candidate_dir = (
    processed_dir
    / "probability_backtests_5min"
    / "lagged_unweighted_long005_short015_timeout_only"
)

candidate_summary = pd.read_csv(candidate_dir / "backtest_summary.csv")
candidate_trades = pd.read_parquet(candidate_dir / "trades_edge_0p050.parquet")

print("Candidate timeout-only backtest summary")
display(candidate_summary)

print("Fold-by-fold performance before kill switch")
fold_summary = (
    candidate_trades
    .groupby("fold")["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        avg_return="mean",
        total_return_sum="sum",
        compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)

display(fold_summary)

print("Side-by-side fold performance")
display(
    candidate_trades
    .groupby(["fold", "side"])["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        avg_return="mean",
        total_return_sum="sum",
        compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)


def apply_fold_kill_switch(
    trades_df,
    max_fold_drawdown=-0.05,
):
    kept_trades = []

    for fold, fold_df in trades_df.groupby("fold", sort=True):
        fold_df = fold_df.sort_values("entry_timestamp").copy()

        equity = 1.0
        peak = 1.0
        stopped = False

        for _, row in fold_df.iterrows():
            if stopped:
                continue

            kept_trades.append(row)

            equity *= 1 + row["net_return"]
            peak = max(peak, equity)

            drawdown = equity / peak - 1

            if drawdown <= max_fold_drawdown:
                stopped = True

    return pd.DataFrame(kept_trades)


kill_switch_results = []

for dd_limit in [-0.03, -0.05, -0.075, -0.10]:
    filtered = apply_fold_kill_switch(
        candidate_trades,
        max_fold_drawdown=dd_limit,
    )

    kill_switch_results.append({
        "max_fold_drawdown": dd_limit,
        "trade_count": len(filtered),
        "win_rate": (filtered["net_return"] > 0).mean(),
        "avg_return": filtered["net_return"].mean(),
        "total_return_sum": filtered["net_return"].sum(),
        "compounded_return": (1 + filtered["net_return"]).prod() - 1,
    })

print("Kill-switch sensitivity")
display(pd.DataFrame(kill_switch_results))

Candidate timeout-only backtest summary


,trade_count,long_trades,short_trades,win_rate,average_return,median_return,average_win,average_loss,profit_factor,total_net_return_compounded,total_net_return_sum,max_drawdown,experiment_name,edge_threshold,min_direction_probability,horizon_bars,take_profit_pct,stop_loss_pct,slippage_bps,fee_bps_per_side,trades_path,long_edge_threshold,short_edge_threshold
0,420,266,154,0.519048,0.000488,0.000555,0.014102,-0.014205,1.071406,0.145757,0.20489,-0.200961,lagged_unweighted_long005_short015_timeout_only,0.05,0.5,12,0.99,0.99,1.0,0.0,/Users/rchbeir/Desktop/Quant work/quant_proj/d...,0.05,0.15


Fold-by-fold performance before kill switch


,fold,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,fold_01,56,0.589286,0.000709,0.039685,0.031465
1,fold_02,103,0.495146,-0.000521,-0.053649,-0.068779
2,fold_03,95,0.547368,0.002217,0.210623,0.213450
3,fold_04,41,0.414634,-0.000687,-0.028179,-0.036092
4,fold_05,20,0.800000,0.007237,0.144731,0.152017
5,fold_06,23,0.434783,0.000600,0.013808,0.010393
6,fold_07,5,0.400000,-0.000589,-0.002946,-0.003209
7,fold_08,26,0.576923,-0.000792,-0.020596,-0.023211
8,fold_09,6,0.500000,-0.001480,-0.008881,-0.009251
9,fold_10,19,0.526316,0.001966,0.037349,0.033875


Side-by-side fold performance


,fold,side,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,fold_01,long,20,0.500000,-0.000886,-0.017725,-0.020448
1,fold_01,short,36,0.638889,0.001595,0.057410,0.052997
2,fold_02,long,88,0.500000,0.000287,0.025263,0.010281
3,fold_02,short,15,0.466667,-0.005261,-0.078913,-0.078255
4,fold_03,long,68,0.544118,0.001667,0.113353,0.105512
5,fold_03,short,27,0.555556,0.003603,0.097270,0.097636
6,fold_04,long,31,0.451613,0.001157,0.035866,0.030326
7,fold_04,short,10,0.300000,-0.006404,-0.064044,-0.064463
8,fold_05,long,12,0.750000,0.006841,0.082089,0.083046
9,fold_05,short,8,0.875000,0.007830,0.062643,0.063683


Kill-switch sensitivity


,max_fold_drawdown,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,-0.030,59,0.474576,-0.002225,-0.131282,-0.131214
1,-0.050,194,0.541237,0.001008,0.195531,0.183145
2,-0.075,263,0.536122,0.000366,0.096343,0.058682
3,-0.100,349,0.535817,0.000600,0.209330,0.167052


## XGBoost Baseline Summary

We used XGBoost as the first baseline model for predicting whether NVDA would move more than 1% up, more than 1% down, or remain neutral within the next 60 minutes.

The final XGBoost feature set included current-candle price and volume features, QQQ/SPY market context, time-of-day features, and lagged/rolling history features. The model was evaluated using expanding walk-forward validation, while the locked final test set was left untouched.

### Classification Results

Two XGBoost versions were tested:

1. Lagged unweighted XGBoost
2. Lagged balanced XGBoost

The unweighted model had better probability quality, meaning it had lower log loss and lower Brier score. However, it predicted the neutral class too often.

The balanced model improved directional recall and slightly improved macro F1, but its probabilities became worse. It predicted more up and down events, but the confidence scores were less reliable.

This created a clear tradeoff:

- The unweighted model was better for probability-based trading signals.
- The balanced model was better at forcing more directional predictions, but worse for calibration.

Because trading decisions depend heavily on probability quality, the lagged unweighted XGBoost model was used as the main model for threshold and backtest experiments.

### Threshold Sweep

Instead of always using the model's highest-probability class, we tested probability threshold rules.

The rule was:

text Long if P(up) - P(down) >= threshold Short if P(down) - P(up) >= threshold Flat otherwise 

We also required the directional probability to be high enough, such as:

text P(up) >= 0.50 for long trades P(down) >= 0.50 for short trades 

The lagged unweighted model produced the best threshold-sweep results. A useful setting was:

text edge_threshold = 0.05 min_direction_probability = 0.50 

This produced a modest directional edge, but the model was still wrong more than half the time. Therefore, the threshold sweep showed that the model had some predictive signal, but it did not prove that the signal was profitable.

### Backtest Results

We then built a simple probability-based backtester. Signals were generated after candle t, entered at the next candle open, and held for up to 12 bars.

Fixed take-profit and stop-loss exits were not profitable. The model performed better when interpreted as a 60-minute directional forecast rather than as a barrier-hit forecast.

The strongest setup used:

text Model: lagged unweighted XGBoost Long edge threshold: 0.05 Short edge threshold: 0.15 Take profit: disabled Stop loss: disabled Exit: 12-bar timeout 

This setup showed positive aggregate performance, but the fold-by-fold results were unstable. Some folds were strongly positive, while others were negative. Fold 11 was especially problematic, suggesting that the model struggled in a different market regime.

### Regime and Risk Control

A simple fold-level drawdown kill switch was tested as a risk-control overlay. The -5% drawdown stop improved the result, but it also removed more than half of the trades.

This suggests that the XGBoost model has a weak positive edge, but the edge is not stable across all regimes. The strategy needs regime detection, risk controls, or a stronger sequence model before it can be considered robust.

### Main Conclusion

The XGBoost baseline is useful, but it is not a finished trading strategy.

It showed:

- Some directional predictive signal.
- Better results with timeout-based exits than with fixed take-profit/stop-loss exits.
- Strong sensitivity to market regime.
- A need for risk controls or regime filters.

The next stage is to build MLP and transformer models and compare them against this XGBoost baseline using the same walk-forward validation setup. The transformer must improve directional recall, probability quality, regime stability, or backtest behavior to justify its added complexity.

In [11]:
from pathlib import Path

import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR
from quant_research.features.feature_builder import MODEL_FEATURES

train_path = (
    Path(PROCESSED_DATA_DIR)
    / "walk_forward_5min"
    / "fold_01"
    / "train.parquet"
)

sample_df = pd.read_parquet(train_path)

print("Rows:", len(sample_df))
print("Features:", len(MODEL_FEATURES))
print("Dates:", sample_df["date"].min(), "to", sample_df["date"].max())
print(sample_df["label_status"].value_counts())

Rows: 49896
Features: 110
Dates: 2016-02-18 to 2021-09-01
label_status
valid                       33249
warmup                      10584
incomplete_future_window     6048
ambiguous_same_bar             15
Name: count, dtype: int64


In [12]:
def build_sequences_from_dataframe(
    df: pd.DataFrame,
    sequence_length: int,
    feature_columns: list[str] = MODEL_FEATURES,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Fast sequence builder.

    Builds sequences within each trading day using numpy sliding windows.

    Context rows may include warmup/non-valid rows, but the final
    target row must have label_status == valid.

    The sequence ends at the target row:
        [t - sequence_length + 1, ..., t]
    """
    all_sequences = []
    all_labels = []
    all_metadata = []

    metadata_columns = [
        "timestamp",
        "timestamp_pt",
        "date",
        "target_name",
        "target_class",
    ]

    for _, day_df in df.groupby("date", sort=False):
        day_df = day_df.sort_values("timestamp").reset_index(drop=True)

        if len(day_df) < sequence_length:
            continue

        features = day_df[feature_columns].to_numpy(dtype=np.float32)

        # Shape:
        # [num_possible_windows, sequence_length, num_features]
        windows = np.lib.stride_tricks.sliding_window_view(
            features,
            window_shape=sequence_length,
            axis=0,
        )

        # sliding_window_view gives:
        # [num_windows, num_features, sequence_length]
        # so we transpose it to:
        # [num_windows, sequence_length, num_features]
        windows = np.transpose(windows, (0, 2, 1))

        # The end row for each window is:
        # sequence_length - 1, sequence_length, ...
        end_positions = np.arange(
            sequence_length - 1,
            len(day_df),
        )

        valid_target_mask = (
            day_df["label_status"].eq("valid")
            & day_df["target_class"].notna()
        ).to_numpy()

        usable_mask = valid_target_mask[end_positions]

        if not usable_mask.any():
            continue

        usable_windows = windows[usable_mask]

        # Drop sequences with NaN/inf values.
        finite_mask = np.isfinite(usable_windows).all(axis=(1, 2))

        if not finite_mask.any():
            continue

        usable_windows = usable_windows[finite_mask]

        usable_end_positions = end_positions[usable_mask][finite_mask]

        labels = (
            day_df
            .loc[usable_end_positions, "target_class"]
            .astype(int)
            .to_numpy(dtype=np.int64)
        )

        metadata = (
            day_df
            .loc[usable_end_positions, metadata_columns]
            .copy()
            .reset_index(drop=True)
        )

        all_sequences.append(usable_windows.astype(np.float32))
        all_labels.append(labels)
        all_metadata.append(metadata)

    if not all_sequences:
        raise ValueError(
            "No valid sequences were built. "
            "Check sequence_length and label_status."
        )

    sequences_array = np.concatenate(all_sequences, axis=0)
    labels_array = np.concatenate(all_labels, axis=0)
    metadata = pd.concat(
        all_metadata,
        axis=0,
        ignore_index=True,
    )

    return sequences_array, labels_array, metadata

In [13]:
import importlib

import quant_research.models.train_sequence_walk_forward as train_sequence_walk_forward

importlib.reload(train_sequence_walk_forward)

transformer_summary_test = (
    train_sequence_walk_forward.train_transformer_walk_forward(
        experiment_name="transformer_seq21_test_1fold",
        timeframe="5min",
        max_folds=1,

        sequence_length=21,
        batch_size=256,

        d_model=64,
        num_heads=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.10,

        learning_rate=1e-3,
        weight_decay=1e-4,
        max_epochs=10,
        patience=3,

        use_class_weights=False,
        device=None,
        random_seed=42,
    )
)

display(transformer_summary_test)

/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_01:  30%|▎| 3/10 [00:06<00:14,  2.01s/epoch, macro_f1=0.4616, val_

fold_01: best_epoch=1, accuracy=0.7133, balanced_accuracy=0.4905, macro_f1=0.5042, log_loss=0.6676

Transformer walk-forward results
   fold  accuracy  balanced_accuracy  macro_f1  weighted_f1  log_loss  multiclass_brier_score  best_epoch
fold_01  0.713254           0.490451  0.504199      0.68708  0.667591                0.376407           1

Mean metrics
number_of_folds: 1
mean_accuracy: 0.7132544036962172
mean_balanced_accuracy: 0.49045073482118
mean_macro_f1: 0.5041994258889013
mean_weighted_f1: 0.6870795141347615
mean_log_loss: 0.6675905995766086
mean_multiclass_brier_score: 0.3764069392563068

Saved results to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/transformer_walk_forward_5min/transformer_seq21_test_1fold


,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score,fold,best_epoch,best_validation_loss,train_sequences,validation_sequences,sequence_length,d_model,num_heads,num_layers,dim_feedforward,dropout,learning_rate,weight_decay,batch_size,use_class_weights,device
0,0.713254,0.490451,0.504199,0.68708,0.667591,0.376407,fold_01,1,0.667591,27709,3463,21,64,4,2,128,0.1,0.001,0.0001,256,False,mps


In [51]:
import importlib

import quant_research.models.train_sequence_walk_forward as train_sequence_walk_forward

importlib.reload(train_sequence_walk_forward)

transformer_summary = (
    train_sequence_walk_forward.train_transformer_walk_forward(
        experiment_name="transformer_seq21_d64_l2_unweighted",
        timeframe="5min",
        max_folds=None,

        sequence_length=21,
        batch_size=256,

        d_model=64,
        num_heads=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.10,

        learning_rate=5e-5,
        weight_decay=1e-6,
        max_epochs=30,
        patience=10,

        use_class_weights=False,
        device=None,
        random_seed=42,
    )
)

display(transformer_summary)

/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_01:  60%|▌| 18/30 [00:13<00:08,  1.34epoch/s, macro_f1=0.4527, val


fold_01: best_epoch=9, accuracy=0.7083, balanced_accuracy=0.4605, macro_f1=0.4718, log_loss=0.6645


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_02:  57%|▌| 17/30 [00:14<00:10,  1.21epoch/s, macro_f1=0.4432, val


fold_02: best_epoch=8, accuracy=0.4902, balanced_accuracy=0.4398, macro_f1=0.4271, log_loss=0.9701


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_03:  50%|▌| 15/30 [00:13<00:13,  1.09epoch/s, macro_f1=0.4415, val


...

/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_10:  40%|▍| 12/30 [00:18<00:28,  1.56s/epoch, macro_f1=0.3733, val


fold_10: best_epoch=3, accuracy=0.7076, balanced_accuracy=0.3828, macro_f1=0.3637, log_loss=0.7442


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_11:  37%|▎| 11/30 [00:18<00:31,  1.67s/epoch, macro_f1=0.4302, val

fold_11: best_epoch=2, accuracy=0.6643, balanced_accuracy=0.4807, macro_f1=0.4897, log_loss=0.8294

Transformer walk-forward results
   fold  accuracy  balanced_accuracy  macro_f1  weighted_f1  log_loss  multiclass_brier_score  best_epoch
fold_01  0.708345           0.460495  0.471775     0.669312  0.664454                0.374527           9
fold_02  0.490185           0.439832  0.427132     0.464866  0.970119                0.581845           8
fold_03  0.473669           0.454286  0.442966     0.455097  1.003137                0.604719           6
fold_04  0.601443           0.387756  0.361364     0.523430  0.911424                0.531806           7
fold_05  0.594400           0.405853  0.379359     0.519393  0.938078                0.545193           5
fold_06  0.725830           0.380039  0.372180     0.646858  0.715686                0.397210           7
fold_07  0.773449           0.347196  0.319194     0.683478  0.630590                0.342497           5
fold_08  0.781241  

,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score,fold,best_epoch,best_validation_loss,train_sequences,validation_sequences,sequence_length,d_model,num_heads,num_layers,dim_feedforward,dropout,learning_rate,weight_decay,batch_size,use_class_weights,device
0,0.708345,0.460495,0.471775,0.669312,0.664454,0.374527,fold_01,9,0.664454,27709,3463,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
1,0.490185,0.439832,0.427132,0.464866,0.970119,0.581845,fold_02,8,0.970119,31172,3464,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
2,0.473669,0.454286,0.442966,0.455097,1.003137,0.604719,fold_03,6,1.003137,34636,3456,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
3,0.601443,0.387756,0.361364,0.523430,0.911424,0.531806,fold_04,7,0.911424,38092,3465,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
4,0.594400,0.405853,0.379359,0.519393,0.938078,0.545193,fold_05,5,0.938078,41557,3464,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
5,0.725830,0.380039,0.372180,0.646858,0.715686,0.397210,fold_06,7,0.715686,45021,3465,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
6,0.773449,0.347196,0.319194,0.683478,0.630590,0.342497,fold_07,5,0.630590,48486,3465,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
7,0.781241,0.357383,0.341957,0.702955,0.640390,0.345882,fold_08,3,0.640390,51951,3465,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
8,0.858297,0.337647,0.318351,0.796599,0.444602,0.225897,fold_09,8,0.444602,55416,3465,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
9,0.707648,0.382792,0.363718,0.622626,0.744181,0.417798,fold_10,3,0.744181,58881,3465,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps


In [50]:
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.models.train_transformer_random_days as train_transformer_random_days

importlib.reload(train_transformer_random_days)

random_day_transformer_summary = (
    train_transformer_random_days.train_transformer_random_day_split(
        experiment_name="transformer_seq21_random_days_d64_l2",

        timeframe="5min",
        processed_dir=PROCESSED_DATA_DIR,

        split_name="random_day_split_5min",
        validation_fraction=0.20,
        test_fraction=0.20,
        random_seed=42,
        recreate_split=True,

        sequence_length=21,
        batch_size=256,

        d_model=64,
        num_heads=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.10,

        learning_rate=5e-5,
        weight_decay=1e-6,

        max_epochs=200,
        patience=30,

        use_class_weights=False,
        device=None,
    )
)

display(random_day_transformer_summary)

Created random-day split
Split directory: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/random_day_split_5min
Train days: 985
Validation days: 246
Locked test days: 308
Train valid targets: 64,997
Validation valid targets: 16,218
Locked test valid targets: 20,320


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_01:  18%|▏| 35/200 [00:49<03:53,  1.41s/epoch, macro_f1=0.4283, va

fold_01: best_epoch=6, accuracy=0.6984, balanced_accuracy=0.4236, macro_f1=0.4286, log_loss=0.7084

Random-day transformer result
   fold  accuracy  balanced_accuracy  macro_f1  weighted_f1  log_loss  multiclass_brier_score  best_epoch
fold_01  0.698365           0.423551  0.428619      0.64702  0.708437                0.399049           6

Saved random-day result to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/transformer_random_days_5min/transformer_seq21_random_days_d64_l2


,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score,fold,best_epoch,best_validation_loss,train_sequences,validation_sequences,sequence_length,d_model,num_heads,num_layers,dim_feedforward,dropout,learning_rate,weight_decay,batch_size,use_class_weights,device
0,0.698365,0.423551,0.428619,0.64702,0.708437,0.399049,fold_01,6,0.708437,54167,13513,21,64,4,2,128,0.1,0.00005,0.000001,256,False,mps


In [52]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

processed_dir = Path(PROCESSED_DATA_DIR)

walk_forward_transformer = pd.read_csv(
    processed_dir
    / "transformer_walk_forward_5min"
    / "transformer_seq21_d64_l2_unweighted"
    / "walk_forward_summary.csv"
)

random_day_transformer = pd.read_csv(
    processed_dir
    / "transformer_random_days_5min"
    / "transformer_seq21_random_days_d64_l2"
    / "random_day_summary.csv"
)

comparison = pd.DataFrame([
    {
        "model": "walk_forward_transformer_mean",
        "accuracy": walk_forward_transformer["accuracy"].mean(),
        "balanced_accuracy": walk_forward_transformer["balanced_accuracy"].mean(),
        "macro_f1": walk_forward_transformer["macro_f1"].mean(),
        "weighted_f1": walk_forward_transformer["weighted_f1"].mean(),
        "log_loss": walk_forward_transformer["log_loss"].mean(),
        "brier": walk_forward_transformer["multiclass_brier_score"].mean(),
    },
    {
        "model": "random_day_transformer",
        "accuracy": random_day_transformer["accuracy"].iloc[0],
        "balanced_accuracy": random_day_transformer["balanced_accuracy"].iloc[0],
        "macro_f1": random_day_transformer["macro_f1"].iloc[0],
        "weighted_f1": random_day_transformer["weighted_f1"].iloc[0],
        "log_loss": random_day_transformer["log_loss"].iloc[0],
        "brier": random_day_transformer["multiclass_brier_score"].iloc[0],
    },
])

display(comparison)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,brier
0,walk_forward_transformer_mean,0.670797,0.403087,0.389790,0.609317,0.772008,0.440394
1,random_day_transformer,0.698365,0.423551,0.428619,0.647020,0.708437,0.399049


In [53]:
from pathlib import Path
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.evaluation.threshold_sweep as threshold_sweep

importlib.reload(threshold_sweep)

random_day_threshold_summary = threshold_sweep.run_threshold_sweep(
    experiment_name="transformer_seq21_random_days_d64_l2",
    results_dir=(
        Path(PROCESSED_DATA_DIR)
        / "transformer_random_days_5min"
        / "transformer_seq21_random_days_d64_l2"
    ),
    output_dir=(
        Path(PROCESSED_DATA_DIR)
        / "threshold_sweep_5min"
        / "transformer_seq21_random_days_d64_l2"
    ),
)

display(random_day_threshold_summary)

Experiment: transformer_seq21_random_days_d64_l2
Loaded predictions: 13,513 rows
Saved summary to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/threshold_sweep_5min/transformer_seq21_random_days_d64_l2/threshold_sweep_summary.csv
Saved fold results to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/threshold_sweep_5min/transformer_seq21_random_days_d64_l2/threshold_sweep_by_fold.csv

Top settings by directional precision:
 edge_threshold  min_direction_probability  signal_rate  long_precision  short_precision  directional_precision  directional_edge_over_base
          0.250                        NaN     0.000074        1.000000              NaN               1.000000                    0.685784
          0.250                       0.30     0.000074        1.000000              NaN               1.000000                    0.685784
          0.250                       0.35     0.000074        1.000000              NaN               1.000000               

,edge_threshold,min_direction_probability,total_rows,long_count,short_count,flat_count,signal_count,long_rate,short_rate,flat_rate,signal_rate,long_precision,short_precision,directional_precision,long_base_rate,short_base_rate,directional_base_rate,long_edge_over_base,short_edge_over_base,directional_edge_over_base,experiment_name
0,0.000,NaN,13513,3676,9837,0,13513,0.272034,0.727966,0.000000,1.000000,0.197225,0.136932,0.153334,0.151854,0.162362,0.314216,0.045371,-0.025430,-0.160882,transformer_seq21_random_days_d64_l2
1,0.025,NaN,13513,1684,4108,7721,5792,0.124621,0.304004,0.571376,0.428624,0.258907,0.192551,0.211844,0.151854,0.162362,0.314216,0.107054,0.030189,-0.102372,transformer_seq21_random_days_d64_l2
2,0.050,NaN,13513,845,1897,10771,2742,0.062532,0.140383,0.797084,0.202916,0.292308,0.247232,0.261123,0.151854,0.162362,0.314216,0.140454,0.084870,-0.053093,transformer_seq21_random_days_d64_l2
3,0.075,NaN,13513,452,934,12127,1386,0.033449,0.069119,0.897432,0.102568,0.296460,0.290150,0.292208,0.151854,0.162362,0.314216,0.144606,0.127788,-0.022008,transformer_seq21_random_days_d64_l2
4,0.100,NaN,13513,225,462,12826,687,0.016651,0.034189,0.949160,0.050840,0.302222,0.279221,0.286754,0.151854,0.162362,0.314216,0.150368,0.116859,-0.027462,transformer_seq21_random_days_d64_l2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,0.150,0.5,13513,26,0,13487,26,0.001924,0.000000,0.998076,0.001924,0.461538,NaN,0.461538,0.151854,0.162362,0.314216,0.309685,NaN,0.147323,transformer_seq21_random_days_d64_l2
62,0.175,0.5,13513,14,0,13499,14,0.001036,0.000000,0.998964,0.001036,0.500000,NaN,0.500000,0.151854,0.162362,0.314216,0.348146,NaN,0.185784,transformer_seq21_random_days_d64_l2
63,0.200,0.5,13513,7,0,13506,7,0.000518,0.000000,0.999482,0.000518,0.428571,NaN,0.428571,0.151854,0.162362,0.314216,0.276718,NaN,0.114355,transformer_seq21_random_days_d64_l2
64,0.250,0.5,13513,1,0,13512,1,0.000074,0.000000,0.999926,0.000074,1.000000,NaN,1.000000,0.151854,0.162362,0.314216,0.848146,NaN,0.685784,transformer_seq21_random_days_d64_l2


In [56]:
from pathlib import Path
import importlib
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.backtesting.probability_backtester as probability_backtester

importlib.reload(probability_backtester)

processed_dir = Path(PROCESSED_DATA_DIR)

# ---------------------------------------------------------
# Transformer experiments to compare
# ---------------------------------------------------------
# Add/remove transformer experiment folders here.

transformer_experiments = {
    "walk_forward_transformer": (
        processed_dir
        / "transformer_walk_forward_5min"
        / "transformer_seq21_d64_l2_unweighted"
    ),
    "continued_transformer": (
        processed_dir
        / "transformer_walk_forward_5min"
        / "transformer_seq21_d64_l2_unweighted_continued"
    ),
    "random_day_transformer": (
        processed_dir
        / "transformer_random_days_5min"
        / "transformer_seq21_random_days_d64_l2"
    ),
}

# ---------------------------------------------------------
# Shared backtest settings
# ---------------------------------------------------------

EDGE_THRESHOLDS = [0.05, 0.10, 0.15]
MIN_DIRECTION_PROBABILITY = 0.45

HORIZON_BARS = 12
TAKE_PROFIT_PCT = 0.99
STOP_LOSS_PCT = 0.99

SLIPPAGE_BPS = 1.0
FEE_BPS_PER_SIDE = 0.0

all_summaries = []

for experiment_name, results_dir in transformer_experiments.items():
    if not results_dir.exists():
        print(f"Skipping missing experiment: {experiment_name}")
        print(results_dir)
        continue

    output_dir = (
        processed_dir
        / "probability_backtests_5min"
        / f"{experiment_name}_long_only_min045_timeout"
    )

    summary = probability_backtester.run_probability_backtest_sweep(
        experiment_name=f"{experiment_name}_long_only_min045_timeout",
        results_dir=results_dir,
        output_dir=output_dir,

        timeframe="5min",
        processed_dir=PROCESSED_DATA_DIR,

        edge_thresholds=EDGE_THRESHOLDS,

        # Long-only rule:
        long_edge_threshold=None,
        short_edge_threshold=99.0,

        min_direction_probability=MIN_DIRECTION_PROBABILITY,
        horizon_bars=HORIZON_BARS,

        take_profit_pct=TAKE_PROFIT_PCT,
        stop_loss_pct=STOP_LOSS_PCT,

        slippage_bps=SLIPPAGE_BPS,
        fee_bps_per_side=FEE_BPS_PER_SIDE,
    )

    summary["model"] = experiment_name
    all_summaries.append(summary)

backtest_comparison = pd.concat(
    all_summaries,
    axis=0,
    ignore_index=True,
)

# ---------------------------------------------------------
# Clean comparison table
# ---------------------------------------------------------

comparison_columns = [
    "model",
    "edge_threshold",
    "trade_count",
    "long_trades",
    "short_trades",
    "win_rate",
    "average_return",
    "average_win",
    "average_loss",
    "profit_factor",
    "total_net_return_compounded",
    "max_drawdown",
]

display(
    backtest_comparison[comparison_columns]
    .sort_values(
        ["edge_threshold", "total_net_return_compounded"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Difference versus baseline walk-forward transformer
# ---------------------------------------------------------
# Baseline = walk_forward_transformer at the same edge threshold.

baseline = (
    backtest_comparison
    .loc[
        backtest_comparison["model"] == "walk_forward_transformer",
        [
            "edge_threshold",
            "trade_count",
            "win_rate",
            "average_return",
            "profit_factor",
            "total_net_return_compounded",
            "max_drawdown",
        ],
    ]
    .copy()
)

baseline = baseline.rename(
    columns={
        "trade_count": "baseline_trade_count",
        "win_rate": "baseline_win_rate",
        "average_return": "baseline_average_return",
        "profit_factor": "baseline_profit_factor",
        "total_net_return_compounded": "baseline_compounded_return",
        "max_drawdown": "baseline_max_drawdown",
    }
)

diff_table = backtest_comparison.merge(
    baseline,
    on="edge_threshold",
    how="left",
)

diff_table["trade_count_diff"] = (
    diff_table["trade_count"]
    - diff_table["baseline_trade_count"]
)

diff_table["win_rate_diff"] = (
    diff_table["win_rate"]
    - diff_table["baseline_win_rate"]
)

diff_table["average_return_diff"] = (
    diff_table["average_return"]
    - diff_table["baseline_average_return"]
)

diff_table["profit_factor_diff"] = (
    diff_table["profit_factor"]
    - diff_table["baseline_profit_factor"]
)

diff_table["compounded_return_diff"] = (
    diff_table["total_net_return_compounded"]
    - diff_table["baseline_compounded_return"]
)

# For drawdown, positive diff means less negative / better.
diff_table["max_drawdown_improvement"] = (
    diff_table["max_drawdown"]
    - diff_table["baseline_max_drawdown"]
)

print("Difference versus walk-forward transformer baseline")
display(
    diff_table[
        [
            "model",
            "edge_threshold",
            "trade_count_diff",
            "win_rate_diff",
            "average_return_diff",
            "profit_factor_diff",
            "compounded_return_diff",
            "max_drawdown_improvement",
        ]
    ]
    .sort_values(
        ["edge_threshold", "compounded_return_diff"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Best row by risk-adjusted behavior
# ---------------------------------------------------------

best_rows = (
    backtest_comparison
    .sort_values(
        ["profit_factor", "total_net_return_compounded"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print("Best transformer backtest rows by profit factor")
display(best_rows[comparison_columns].head(10))

Experiment: walk_forward_transformer_long_only_min045_timeout
Saved backtest summary to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/probability_backtests_5min/walk_forward_transformer_long_only_min045_timeout/backtest_summary.csv

Backtest summary:
 edge_threshold  trade_count  long_trades  short_trades  win_rate  average_return  average_win  average_loss  profit_factor  total_net_return_compounded  max_drawdown
           0.05          137          137             0  0.532847        0.001219     0.015018     -0.014521       1.179642                     0.154132     -0.213815
           0.10           77           77             0  0.610390        0.002545     0.013469     -0.014570       1.448313                     0.202619     -0.074738
           0.15           31           31             0  0.612903        0.003626     0.013956     -0.012730       1.735859                     0.114024     -0.057404
Experiment: continued_transformer_long_only_min045_timeout
Saved b

,model,edge_threshold,trade_count,long_trades,short_trades,win_rate,average_return,average_win,average_loss,profit_factor,total_net_return_compounded,max_drawdown
0,continued_transformer,0.05,138,138,0,0.536232,0.001214,0.015031,-0.014763,1.177268,0.154416,-0.211124
1,walk_forward_transformer,0.05,137,137,0,0.532847,0.001219,0.015018,-0.014521,1.179642,0.154132,-0.213815
2,random_day_transformer,0.05,67,67,0,0.507463,0.000924,0.012341,-0.010839,1.173105,0.055402,-0.100631
3,continued_transformer,0.10,78,78,0,0.641026,0.003343,0.013902,-0.015511,1.600391,0.281835,-0.065727
4,walk_forward_transformer,0.10,77,77,0,0.610390,0.002545,0.013469,-0.014570,1.448313,0.202619,-0.074738
5,random_day_transformer,0.10,45,45,0,0.533333,0.000915,0.011391,-0.011058,1.177257,0.036832,-0.088122
6,continued_transformer,0.15,32,32,0,0.625000,0.003876,0.013604,-0.012337,1.837761,0.127076,-0.057404
7,walk_forward_transformer,0.15,31,31,0,0.612903,0.003626,0.013956,-0.012730,1.735859,0.114024,-0.057404
8,random_day_transformer,0.15,21,21,0,0.523810,-0.002235,0.006958,-0.012347,0.619878,-0.047483,-0.085684


Difference versus walk-forward transformer baseline


,model,edge_threshold,trade_count_diff,win_rate_diff,average_return_diff,profit_factor_diff,compounded_return_diff,max_drawdown_improvement
0,continued_transformer,0.05,1,0.003385,-0.000005,-0.002374,0.000284,2.691535e-03
1,walk_forward_transformer,0.05,0,0.000000,0.000000,0.000000,0.000000,0.000000e+00
2,random_day_transformer,0.05,-70,-0.025384,-0.000294,-0.006537,-0.098730,1.131848e-01
3,continued_transformer,0.10,1,0.030636,0.000798,0.152078,0.079216,9.011051e-03
4,walk_forward_transformer,0.10,0,0.000000,0.000000,0.000000,0.000000,0.000000e+00
5,random_day_transformer,0.10,-32,-0.077056,-0.001630,-0.271056,-0.165787,-1.338351e-02
6,continued_transformer,0.15,1,0.012097,0.000250,0.101902,0.013052,-2.220446e-16
7,walk_forward_transformer,0.15,0,0.000000,0.000000,0.000000,0.000000,0.000000e+00
8,random_day_transformer,0.15,-10,-0.089094,-0.005861,-1.115981,-0.161507,-2.827926e-02


Best transformer backtest rows by profit factor


,model,edge_threshold,trade_count,long_trades,short_trades,win_rate,average_return,average_win,average_loss,profit_factor,total_net_return_compounded,max_drawdown
0,continued_transformer,0.15,32,32,0,0.625000,0.003876,0.013604,-0.012337,1.837761,0.127076,-0.057404
1,walk_forward_transformer,0.15,31,31,0,0.612903,0.003626,0.013956,-0.012730,1.735859,0.114024,-0.057404
2,continued_transformer,0.10,78,78,0,0.641026,0.003343,0.013902,-0.015511,1.600391,0.281835,-0.065727
3,walk_forward_transformer,0.10,77,77,0,0.610390,0.002545,0.013469,-0.014570,1.448313,0.202619,-0.074738
4,walk_forward_transformer,0.05,137,137,0,0.532847,0.001219,0.015018,-0.014521,1.179642,0.154132,-0.213815
5,continued_transformer,0.05,138,138,0,0.536232,0.001214,0.015031,-0.014763,1.177268,0.154416,-0.211124
6,random_day_transformer,0.10,45,45,0,0.533333,0.000915,0.011391,-0.011058,1.177257,0.036832,-0.088122
7,random_day_transformer,0.05,67,67,0,0.507463,0.000924,0.012341,-0.010839,1.173105,0.055402,-0.100631
8,random_day_transformer,0.15,21,21,0,0.523810,-0.002235,0.006958,-0.012347,0.619878,-0.047483,-0.085684


In [57]:
BEST_TRANSFORMER_EXPERIMENT = "transformer_seq21_d64_l2_unweighted_continued"

BEST_RULE = {
    "side": "long_only",
    "min_direction_probability": 0.45,
    "edge_threshold": 0.10,
    "horizon_bars": 12,
    "take_profit_pct": 0.99,
    "stop_loss_pct": 0.99,
    "slippage_bps": 1.0,
    "fee_bps_per_side": 0.0,
}

In [58]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

processed_dir = Path(PROCESSED_DATA_DIR)

trades_path = (
    processed_dir
    / "probability_backtests_5min"
    / "continued_transformer_long_only_min045_timeout"
    / "trades_edge_0p100.parquet"
)

trades = pd.read_parquet(trades_path)

print("Loaded trades:", len(trades))
display(trades.head())

fold_summary = (
    trades
    .groupby("fold")["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        avg_return="mean",
        total_return_sum="sum",
        compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)

display(fold_summary)

print("Positive folds:")
print(
    (fold_summary["total_return_sum"] > 0).sum(),
    "out of",
    len(fold_summary),
)

print("\nOverall:")
overall = pd.DataFrame([
    {
        "trade_count": len(trades),
        "win_rate": (trades["net_return"] > 0).mean(),
        "avg_return": trades["net_return"].mean(),
        "total_return_sum": trades["net_return"].sum(),
        "compounded_return": (1 + trades["net_return"]).prod() - 1,
        "max_drawdown": (
            ((1 + trades["net_return"]).cumprod()
             / (1 + trades["net_return"]).cumprod().cummax())
            - 1
        ).min(),
    }
])

display(overall)

Loaded trades: 78


,fold,date,signal_timestamp,signal_timestamp_pt,entry_timestamp,entry_timestamp_pt,exit_timestamp,exit_timestamp_pt,signal_index,entry_index,exit_index,side,entry_price,exit_price,raw_entry_price,raw_exit_price,gross_return,net_return,exit_reason,p_neutral,p_up,p_down,signal_edge,target_name,target_class,bars_held,edge_threshold,min_direction_probability,horizon_bars,take_profit_pct,stop_loss_pct,slippage_bps,fee_bps_per_side
0,fold_01,2021-11-04,2021-11-04 15:35:00+00:00,2021-11-04 08:35:00-07:00,2021-11-04 15:40:00+00:00,2021-11-04 08:40:00-07:00,2021-11-04 16:35:00+00:00,2021-11-04 09:35:00-07:00,53209,53210,53221,long,289.498947,294.900507,289.47,294.9300,0.018658,0.018658,timeout,0.144271,0.487334,0.368395,0.118940,up,1,12,0.1,0.45,12,0.99,0.99,1.0,0.0
1,fold_01,2021-11-04,2021-11-04 18:50:00+00:00,2021-11-04 11:50:00-07:00,2021-11-04 18:55:00+00:00,2021-11-04 11:55:00-07:00,2021-11-04 19:50:00+00:00,2021-11-04 12:50:00-07:00,53248,53249,53260,long,309.820979,297.170280,309.79,297.2000,-0.040832,-0.040832,timeout,0.153136,0.483563,0.363301,0.120262,down,2,12,0.1,0.45,12,0.99,0.99,1.0,0.0
2,fold_01,2021-11-05,2021-11-05 14:25:00+00:00,2021-11-05 07:25:00-07:00,2021-11-05 14:30:00+00:00,2021-11-05 07:30:00-07:00,2021-11-05 15:25:00+00:00,2021-11-05 08:25:00-07:00,53294,53295,53306,long,307.660763,309.938903,307.63,309.9699,0.007405,0.007405,timeout,0.161191,0.491855,0.346954,0.144900,up,1,12,0.1,0.45,12,0.99,0.99,1.0,0.0
3,fold_01,2021-11-05,2021-11-05 17:35:00+00:00,2021-11-05 10:35:00-07:00,2021-11-05 17:40:00+00:00,2021-11-05 10:40:00-07:00,2021-11-05 18:35:00+00:00,2021-11-05 11:35:00-07:00,53332,53333,53344,long,301.210118,301.418655,301.18,301.4488,0.000692,0.000692,timeout,0.202817,0.458224,0.338959,0.119265,neutral,0,12,0.1,0.45,12,0.99,0.99,1.0,0.0
4,fold_01,2021-12-14,2021-12-14 15:25:00+00:00,2021-12-14 07:25:00-08:00,2021-12-14 15:30:00+00:00,2021-12-14 07:30:00-08:00,2021-12-14 16:25:00+00:00,2021-12-14 08:25:00-08:00,55769,55770,55781,long,284.728470,280.471950,284.70,280.5000,-0.014949,-0.014949,timeout,0.093709,0.540056,0.366234,0.173822,down,2,12,0.1,0.45,12,0.99,0.99,1.0,0.0


,fold,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,fold_01,6,0.666667,-0.003474,-0.020841,-0.021803
1,fold_02,20,0.650000,0.003449,0.068983,0.068331
2,fold_03,29,0.655172,0.004090,0.118609,0.120218
3,fold_04,2,1.000000,0.015763,0.031525,0.031548
4,fold_05,4,0.500000,-0.010294,-0.041174,-0.040962
5,fold_06,5,0.600000,0.005599,0.027995,0.027648
6,fold_07,1,1.000000,0.016282,0.016282,0.016282
7,fold_09,1,1.000000,0.000215,0.000215,0.000215
8,fold_10,4,0.000000,-0.005751,-0.023003,-0.022822
9,fold_11,6,0.833333,0.013695,0.082172,0.084292


Positive folds:
7 out of 10

Overall:


,trade_count,win_rate,avg_return,total_return_sum,compounded_return,max_drawdown
0,78,0.641026,0.003343,0.26076,0.281835,-0.065727


In [59]:
daily_summary = (
    trades
    .groupby("date")["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        daily_return_sum="sum",
        daily_compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)

print("Number of trading days with trades:", len(daily_summary))

display(
    daily_summary
    .sort_values("daily_compounded_return", ascending=False)
    .head(10)
)

display(
    daily_summary
    .sort_values("daily_compounded_return", ascending=True)
    .head(10)
)

print("Positive days:")
print(
    (daily_summary["daily_return_sum"] > 0).sum(),
    "out of",
    len(daily_summary),
)

Number of trading days with trades: 63


,date,trade_count,win_rate,daily_return_sum,daily_compounded_return
11,2022-01-28,1,1.00,0.033361,0.033361
40,2022-06-15,1,1.00,0.032890,0.032890
48,2023-02-01,1,1.00,0.031028,0.031028
42,2022-09-21,1,1.00,0.030773,0.030773
7,2022-01-24,4,0.75,0.031251,0.030687
28,2022-05-06,2,0.50,0.028099,0.027954
27,2022-05-04,1,1.00,0.026296,0.026296
20,2022-04-06,1,1.00,0.025774,0.025774
60,2024-08-02,2,0.50,0.021000,0.020429
61,2024-08-05,1,1.00,0.017873,0.017873


,date,trade_count,win_rate,daily_return_sum,daily_compounded_return
35,2022-05-20,2,0.0,-0.036409,-0.036168
9,2022-01-26,1,0.0,-0.030771,-0.030771
44,2022-11-02,1,0.0,-0.030211,-0.030211
0,2021-11-04,2,0.5,-0.022174,-0.022936
46,2022-11-17,1,0.0,-0.017903,-0.017903
30,2022-05-11,1,0.0,-0.016332,-0.016332
4,2021-12-20,1,0.0,-0.015891,-0.015891
52,2023-03-22,1,0.0,-0.015883,-0.015883
31,2022-05-12,2,0.5,-0.015538,-0.015643
2,2021-12-14,1,0.0,-0.014949,-0.014949


Positive days:
42 out of 63


In [60]:
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.models.train_transformer_locked_test as train_transformer_locked_test

importlib.reload(train_transformer_locked_test)

locked_test_summary = (
    train_transformer_locked_test.train_final_transformer_locked_test(
        experiment_name="final_transformer_seq21_long_candidate",

        timeframe="5min",
        processed_dir=PROCESSED_DATA_DIR,

        split_name="locked_test_transformer_split_5min",
        recreate_split=True,

        test_fraction=0.20,
        early_stopping_fraction=0.10,

        sequence_length=21,
        batch_size=256,

        d_model=64,
        num_heads=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.10,

        learning_rate=5e-5,
        weight_decay=1e-6,

        max_epochs=30,
        patience=5,

        continue_training=True,
        continue_learning_rate=2e-5,
        continue_weight_decay=1e-5,
        continue_epochs=10,
        continue_patience=3,

        device=None,
        random_seed=42,
    )
)

display(locked_test_summary)

Created locked-test transformer split
{
  "timeframe": "5min",
  "test_fraction": 0.2,
  "early_stopping_fraction": 0.1,
  "total_days": 1539,
  "train_days": 1107,
  "early_stop_days": 124,
  "locked_test_days": 308,
  "train_start": "2016-02-18",
  "train_end": "2024-04-04",
  "early_stop_start": "2024-04-05",
  "early_stop_end": "2024-10-04",
  "locked_test_start": "2024-10-07",
  "locked_test_end": "2026-01-02",
  "train_rows": 109593,
  "early_stop_rows": 12276,
  "locked_test_rows": 30492,
  "train_valid_targets": 73033,
  "early_stop_valid_targets": 8182,
  "locked_test_valid_targets": 20320
}


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


stage_1 epoch 1: train_loss=0.8165, early_loss=0.8168, macro_f1=0.4480
stage_1 epoch 2: train_loss=0.7117, early_loss=0.8039, macro_f1=0.4436
stage_1 epoch 3: train_loss=0.7033, early_loss=0.8014, macro_f1=0.4376
stage_1 epoch 4: train_loss=0.6991, early_loss=0.8122, macro_f1=0.4348
stage_1 epoch 5: train_loss=0.6953, early_loss=0.8077, macro_f1=0.4054
stage_1 epoch 6: train_loss=0.6926, early_loss=0.8071, macro_f1=0.4232
stage_1 epoch 7: train_loss=0.6909, early_loss=0.8011, macro_f1=0.4029
stage_1 epoch 8: train_loss=0.6893, early_loss=0.8048, macro_f1=0.4128
stage_1 epoch 9: train_loss=0.6887, early_loss=0.8066, macro_f1=0.4145
stage_1 epoch 10: train_loss=0.6874, early_loss=0.8108, macro_f1=0.4165
stage_1 epoch 11: train_loss=0.6855, early_loss=0.8061, macro_f1=0.4100
stage_1 epoch 12: train_loss=0.6849, early_loss=0.8066, macro_f1=0.4133
stage_2_continue epoch 1: train_loss=0.6892, early_loss=0.8038, macro_f1=0.4078
stage_2_continue epoch 2: train_loss=0.6880, early_loss=0.8091, m

,experiment_name,best_early_stop_loss,stage_1_best_epoch,stage_2_continued_best_epoch,train_sequences,early_stop_sequences,locked_test_sequences,sequence_length,learning_rate,weight_decay,continue_training,continue_learning_rate,continue_weight_decay,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score
0,final_transformer_seq21_long_candidate,0.801069,7,0,60861,6819,16936,21,0.00005,0.000001,True,0.00002,0.00001,0.790092,0.416022,0.419463,0.731971,0.597385,0.319536


In [61]:
from pathlib import Path
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.backtesting.probability_backtester as probability_backtester

importlib.reload(probability_backtester)

locked_test_backtest = probability_backtester.run_probability_backtest_sweep(
    experiment_name="final_transformer_locked_test_long_only_min045_edge010",
    results_dir=(
        Path(PROCESSED_DATA_DIR)
        / "transformer_locked_test_5min"
        / "final_transformer_seq21_long_candidate"
    ),
    output_dir=(
        Path(PROCESSED_DATA_DIR)
        / "probability_backtests_5min"
        / "final_transformer_locked_test_long_only_min045_edge010"
    ),

    timeframe="5min",
    processed_dir=PROCESSED_DATA_DIR,

    edge_thresholds=[0.10],
    long_edge_threshold=None,
    short_edge_threshold=99.0,

    min_direction_probability=0.45,
    horizon_bars=12,

    take_profit_pct=0.99,
    stop_loss_pct=0.99,

    slippage_bps=1.0,
    fee_bps_per_side=0.0,
)

display(locked_test_backtest)

Experiment: final_transformer_locked_test_long_only_min045_edge010
Saved backtest summary to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/probability_backtests_5min/final_transformer_locked_test_long_only_min045_edge010/backtest_summary.csv

Backtest summary:
 edge_threshold  trade_count  long_trades  short_trades  win_rate  average_return  average_win  average_loss  profit_factor  total_net_return_compounded  max_drawdown
            0.1           33           33             0  0.484848        0.000828     0.015768     -0.013232       1.121513                     0.022348     -0.071643


,trade_count,long_trades,short_trades,win_rate,average_return,median_return,average_win,average_loss,profit_factor,total_net_return_compounded,total_net_return_sum,max_drawdown,experiment_name,edge_threshold,min_direction_probability,horizon_bars,take_profit_pct,stop_loss_pct,slippage_bps,fee_bps_per_side,trades_path,long_edge_threshold,short_edge_threshold
0,33,33,0,0.484848,0.000828,-0.002238,0.015768,-0.013232,1.121513,0.022348,0.027335,-0.071643,final_transformer_locked_test_long_only_min045...,0.1,0.45,12,0.99,0.99,1.0,0.0,/Users/rchbeir/Desktop/Quant work/quant_proj/d...,None,99.0


In [62]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

trades_path = (
    Path(PROCESSED_DATA_DIR)
    / "probability_backtests_5min"
    / "final_transformer_locked_test_long_only_min045_edge010"
    / "trades_edge_0p100.parquet"
)

locked_trades = pd.read_parquet(trades_path)

locked_trades["entry_timestamp_pt"] = pd.to_datetime(
    locked_trades["entry_timestamp_pt"]
)

locked_trades["month"] = locked_trades["entry_timestamp_pt"].dt.to_period("M")

monthly_summary = (
    locked_trades
    .groupby("month")["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        avg_return="mean",
        total_return_sum="sum",
        compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)

display(monthly_summary)

print("Positive months:")
print(
    (monthly_summary["total_return_sum"] > 0).sum(),
    "out of",
    len(monthly_summary),
)

display(
    locked_trades[
        [
            "date",
            "side",
            "net_return",
            "p_neutral",
            "p_up",
            "p_down",
            "signal_edge",
            "target_name",
            "bars_held",
        ]
    ]
    .sort_values("net_return")
)

/var/folders/l3/58vmcgtx0x5cs_vsgm5lp5j00000gn/T/ipykernel_9201/121110016.py:19: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  locked_trades["month"] = locked_trades["entry_timestamp_pt"].dt.to_period("M")


,month,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,2024-11,1,1.000000,0.003256,0.003256,0.003256
1,2024-12,1,0.000000,-0.013527,-0.013527,-0.013527
2,2025-01,4,0.000000,-0.014957,-0.059829,-0.058913
3,2025-02,2,1.000000,0.005463,0.010926,0.010954
4,2025-03,5,0.800000,0.009979,0.049893,0.049648
5,2025-04,16,0.500000,0.002546,0.040730,0.039317
6,2025-10,1,0.000000,-0.008756,-0.008756,-0.008756
7,2025-11,3,0.333333,0.001547,0.004642,0.004080


Positive months:
5 out of 8


,date,side,net_return,p_neutral,p_up,p_down,signal_edge,target_name,bars_held
4,2025-01-27,long,-0.039541,0.149096,0.486829,0.364075,0.122754,down,12
10,2025-03-07,long,-0.031130,0.114001,0.498887,0.387111,0.111776,down,12
14,2025-04-04,long,-0.018652,0.080258,0.546535,0.373206,0.173329,down,12
30,2025-11-20,long,-0.016595,0.097574,0.509212,0.393215,0.115997,down,12
15,2025-04-04,long,-0.015261,0.149509,0.496127,0.354363,0.141764,down,12
1,2024-12-18,long,-0.013527,0.194627,0.454744,0.350630,0.104114,down,12
22,2025-04-08,long,-0.012992,0.079420,0.545359,0.375221,0.170138,down,12
26,2025-04-10,long,-0.010880,0.127818,0.501322,0.370860,0.130462,up,12
3,2025-01-10,long,-0.010672,0.189147,0.471805,0.339049,0.132756,down,12
20,2025-04-08,long,-0.010039,0.113371,0.514752,0.371878,0.142874,up,12


# Model Progress Summary: XGBoost → Transformer → Locked Test → HMM Next Step

## 1. Feature Engineering and Dataset Setup

We started by building a model-safe feature dataset from the labeled 5-minute NVDA / QQQ / SPY data. The feature set included:

- NVDA price-action features: body %, candle range %, upper/lower wick %, close position, VWAP distance.
- Activity features: log volume, log trade count, log average trade size, log dollar volume.
- One-bar lag features: log return, volume change, trade count change.
- QQQ and SPY equivalents.
- Cross-asset features: NVDA minus QQQ return, NVDA minus SPY return, QQQ minus SPY return.
- Rolling lag features over 3, 6, and 12 bars.
- Time features: circular time-of-day encoding, session position, circular day-of-week encoding.

The final model feature dataset had 110 approved features. Rows with `label_status == "valid"` were used as supervised prediction targets, while warmup / incomplete rows were preserved so sequence models could use them as context.

---

## 2. XGBoost Baseline

We first trained XGBoost using expanding walk-forward validation. Each fold trained on all previous dates and validated on the next chronological block.

The unweighted XGBoost model had reasonable overall accuracy, but it was heavily biased toward the neutral class. It performed better than random on some directional predictions, but still struggled with `up` and `down` recall.

A balanced-weight XGBoost version improved directional balance somewhat, but worsened probability calibration and backtest behavior.

The key lesson from XGBoost was:

> XGBoost could produce usable directional signals, but the strategy was regime-sensitive and suffered from unstable drawdowns.

Backtesting XGBoost showed that some configurations could generate positive validation returns, but the results were unstable across folds. Certain folds, especially later market regimes, hurt performance badly. This suggested that a more sequence-aware model might help.

---

## 3. Transformer Sequence Model

We then built a transformer sequence classifier. Instead of predicting from one candle at a time, the transformer received a sequence of recent candles.

The first transformer used:

- Sequence length: 21 candles
- Feature count: 110
- Architecture: 2-layer transformer encoder
- Model dimension: 64
- Attention heads: 4
- Feedforward dimension: 128
- Dropout: 0.10
- Loss: cross entropy
- Device: MPS on Mac

The transformer had better probability quality than XGBoost, with lower log loss and lower Brier score, but worse macro F1 and balanced accuracy. This showed that the transformer learned cleaner probabilities but was even more conservative / neutral-biased as an argmax classifier.

The key insight was:

> The transformer was not better as a general classifier, but it could be useful as a high-confidence sparse signal generator.

---

## 4. Transformer Threshold Sweep and Backtesting

We tested transformer probabilities using a threshold-based signal rule:

- Long signal when `p_up >= min_direction_probability` and `p_up - p_down >= edge_threshold`.
- Short signal disabled later because short signals were weaker.
- Exit rule: 12-bar timeout.
- Take-profit and stop-loss effectively disabled using very large values.
- Slippage: 1 bp.

The best validation transformer behavior came from long-only signals.

The strongest validation setup was:

- Model: continued transformer
- Side: long-only
- `min_direction_probability = 0.45`
- `edge_threshold = 0.10`
- Horizon: 12 bars
- Exit: timeout only
- Slippage: 1 bp

Validation result:

- Trades: 78
- Win rate: 64.10%
- Average return: +0.3343%
- Profit factor: 1.60
- Compounded return: +28.18%
- Max drawdown: -6.57%

Fold stability was much better than XGBoost:

- Positive folds: 7 out of 10 active folds
- Positive trading days: 42 out of 63 active days
- No single catastrophic fold

This suggested that the transformer was more useful as a sparse high-confidence long model than as a full long/short classifier.

---

## 5. Random-Day Diagnostic

We also trained a random-day transformer split to test whether the walk-forward model was suffering mainly from chronological regime shift.

The random-day model did not outperform the walk-forward transformer. Its backtest results were weaker, especially compared to the continued walk-forward transformer.

This suggested that simply mixing market regimes randomly was not enough. The sequence model still worked best when trained and validated chronologically.

---

## 6. Locked-Test Evaluation

After selecting the best validation rule, we stopped tuning and moved to the locked test.

The locked test period was:

- Start: 2024-10-07
- End: 2026-01-02

The final transformer was trained on the development period with an early-stopping window before the locked test. The same fixed long-only rule was then applied once to the locked test.

Locked-test result:

- Trades: 33
- Win rate: 48.48%
- Average return: +0.0828%
- Profit factor: 1.12
- Compounded return: +2.23%
- Max drawdown: -7.16%

The locked-test result was much weaker than validation. The validation edge did not fully generalize. However, the strategy did not completely fail: it remained slightly positive with controlled drawdown.

The key locked-test conclusion:

> The transformer learned useful probability structure, but the strong validation trading edge was partly regime-specific or overfit. The locked-test result is weakly positive, not strong enough to call a robust strategy yet.

---

## 7. Locked-Test Monthly Breakdown

The locked-test trades were spread across 8 active months:

- Positive months: 5 out of 8
- Best months: March 2025 and April 2025
- Worst month: January 2025

The return was not entirely from a single trade, but performance was uneven. January 2025 was especially poor, while March and April carried most of the positive result.

This suggests that regime awareness may be important.

---

## 8. Why HMM Is the Next Step

The next step is to add a Hidden Markov Model as a regime feature for the transformer.

The goal is not to use HMM as a separate post-trade filter yet. Instead, the plan is:

> Fit an HMM on market behavior, infer the current regime, and feed that regime information into the transformer as extra input features.

Possible HMM-derived features:

- HMM state ID
- One-hot encoded HMM state
- HMM posterior probabilities for each regime
- Regime return rank
- Regime volatility rank
- Regime transition probability / confidence

The purpose is to help the transformer learn patterns like:

- This long signal works better in low-volatility upward regimes.
- This signal fails in high-volatility selloff regimes.
- The same technical setup may mean different things in different market states.

Most importantly, the HMM must be fit without leakage:

- For walk-forward validation, each fold’s HMM should be fit only on that fold’s training data.
- The HMM should then infer regimes on that fold’s validation data.
- For locked test, the HMM should be fit only on the final training/development data, then applied once to the locked test.

The next experiment will be:

> Build HMM regime features, append them to the transformer feature set, retrain the transformer, and compare whether regime-aware transformer probabilities improve validation and locked-test backtest performance.

In [72]:
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.regime.hmm_feature_builder as hmm_feature_builder

importlib.reload(hmm_feature_builder)

hmm_walk_forward_dir = (
    hmm_feature_builder.build_hmm_augmented_walk_forward_splits(
        timeframe="5min",
        processed_dir=PROCESSED_DATA_DIR,
        output_name="walk_forward_hmm_5min",
        number_of_states=3,
        random_seed=42,
        covariance_type="diag",
    )
)

print(hmm_walk_forward_dir)

Building HMM features for fold_01
Building HMM features for fold_02
Building HMM features for fold_03
Building HMM features for fold_04
Building HMM features for fold_05
Building HMM features for fold_06
Building HMM features for fold_07
Building HMM features for fold_08
Building HMM features for fold_09
Building HMM features for fold_10
Building HMM features for fold_11

Saved HMM-augmented splits to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/walk_forward_hmm_5min
/Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/walk_forward_hmm_5min


In [82]:
from pathlib import Path
import importlib

from quant_research.features.feature_builder import MODEL_FEATURES
from quant_research.regime.hmm_feature_builder import HMM_OUTPUT_FEATURES_3_STATE
from quant_research.utils.paths import PROCESSED_DATA_DIR

import quant_research.models.train_sequence_walk_forward as train_sequence_walk_forward

importlib.reload(train_sequence_walk_forward)

MODEL_FEATURES_HMM = MODEL_FEATURES + HMM_OUTPUT_FEATURES_3_STATE

hmm_transformer_summary = train_sequence_walk_forward.train_transformer_walk_forward(
    experiment_name="transformer_seq21_d64_l2_hmm_features_full",
    timeframe="5min",
    processed_dir=PROCESSED_DATA_DIR,
    walk_forward_directory=(
        Path(PROCESSED_DATA_DIR) / "walk_forward_hmm_5min"
    ),

    max_folds=None,

    sequence_length=21,
    batch_size=256,

    d_model=64,
    num_heads=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.10,

    learning_rate=5e-5,
    weight_decay=1e-6,

    max_epochs=30,
    patience=5,

    use_class_weights=False,
    device=None,
    random_seed=42,

    feature_columns=MODEL_FEATURES_HMM,
)

display(hmm_transformer_summary)

/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_01:  47%|▍| 14/30 [00:12<00:13,  1.16epoch/s, macro_f1=0.4204, val


fold_01: training with 116 features
fold_01: best_epoch=10, accuracy=0.7026, balanced_accuracy=0.4325, macro_f1=0.4178, log_loss=0.6707


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_02:  37%|▎| 11/30 [00:10<00:17,  1.07epoch/s, macro_f1=0.4077, val


fold_02: training with 116 features
fold_02: best_epoch=7, accuracy=0.4876, balanced_accuracy=0.4359, macro_f1=0.4038, log_loss=0.9755


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_03:  37%|▎| 11/30 [00:10<00:18,  1.01epoch/s, macro_f1=0.4386, val


...

/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_10:  23%|▏| 7/30 [00:11<00:37,  1.64s/epoch, macro_f1=0.3672, val_


fold_10: training with 116 features
fold_10: best_epoch=3, accuracy=0.7094, balanced_accuracy=0.3729, macro_f1=0.3556, log_loss=0.7489


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
Training fold_11:  30%|▎| 9/30 [00:15<00:36,  1.72s/epoch, macro_f1=0.4406, val_

fold_11: training with 116 features
fold_11: best_epoch=5, accuracy=0.6510, balanced_accuracy=0.4482, macro_f1=0.4475, log_loss=0.8191

Transformer walk-forward results
   fold  accuracy  balanced_accuracy  macro_f1  weighted_f1  log_loss  multiclass_brier_score  best_epoch  num_features
fold_01  0.702570           0.432478  0.417798     0.641519  0.670704                0.378295          10           116
fold_02  0.487587           0.435934  0.403764     0.444890  0.975492                0.583875           7           116
fold_03  0.473669           0.450611  0.433544     0.449232  1.002490                0.603395           7           116
fold_04  0.601443           0.405760  0.384689     0.537570  0.913420                0.532942           4           116
fold_05  0.590935           0.405346  0.385460     0.523784  0.935920                0.543941          10           116
fold_06  0.724387           0.380276  0.369539     0.645329  0.713480                0.396269           6      

,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score,fold,best_epoch,best_validation_loss,train_sequences,validation_sequences,sequence_length,num_features,d_model,num_heads,num_layers,dim_feedforward,dropout,learning_rate,weight_decay,batch_size,use_class_weights,device
0,0.702570,0.432478,0.417798,0.641519,0.670704,0.378295,fold_01,10,0.670704,27709,3463,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
1,0.487587,0.435934,0.403764,0.444890,0.975492,0.583875,fold_02,7,0.975492,31172,3464,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
2,0.473669,0.450611,0.433544,0.449232,1.002490,0.603395,fold_03,7,1.002490,34636,3456,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
3,0.601443,0.405760,0.384689,0.537570,0.913420,0.532942,fold_04,4,0.913420,38092,3465,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
4,0.590935,0.405346,0.385460,0.523784,0.935920,0.543941,fold_05,10,0.935920,41557,3464,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
5,0.724387,0.380276,0.369539,0.645329,0.713480,0.396269,fold_06,6,0.713480,45021,3465,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
6,0.773737,0.346473,0.317931,0.684345,0.634270,0.344734,fold_07,6,0.634270,48486,3465,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
7,0.781530,0.356358,0.336898,0.700103,0.641299,0.346674,fold_08,3,0.641299,51951,3465,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
8,0.857720,0.334890,0.312838,0.794990,0.446685,0.227007,fold_09,7,0.446685,55416,3465,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps
9,0.709380,0.372851,0.355554,0.623690,0.748938,0.420219,fold_10,3,0.748938,58881,3465,21,116,64,4,2,128,0.1,0.00005,0.000001,256,False,mps


In [83]:
from pathlib import Path
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.backtesting.probability_backtester as probability_backtester

importlib.reload(probability_backtester)

hmm_transformer_backtest = probability_backtester.run_probability_backtest_sweep(
    experiment_name="transformer_hmm_features_full_long_only_min045_timeout",
    results_dir=(
        Path(PROCESSED_DATA_DIR)
        / "transformer_walk_forward_5min"
        / "transformer_seq21_d64_l2_hmm_features_full"
    ),
    output_dir=(
        Path(PROCESSED_DATA_DIR)
        / "probability_backtests_5min"
        / "transformer_hmm_features_full_long_only_min045_timeout"
    ),

    timeframe="5min",
    processed_dir=PROCESSED_DATA_DIR,

    edge_thresholds=[0.05, 0.10, 0.15],
    long_edge_threshold=None,
    short_edge_threshold=99.0,

    min_direction_probability=0.45,
    horizon_bars=12,

    take_profit_pct=0.99,
    stop_loss_pct=0.99,

    slippage_bps=1.0,
    fee_bps_per_side=0.0,
)

display(hmm_transformer_backtest)

Experiment: transformer_hmm_features_full_long_only_min045_timeout
Saved backtest summary to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/probability_backtests_5min/transformer_hmm_features_full_long_only_min045_timeout/backtest_summary.csv

Backtest summary:
 edge_threshold  trade_count  long_trades  short_trades  win_rate  average_return  average_win  average_loss  profit_factor  total_net_return_compounded  max_drawdown
           0.05          105          105             0  0.533333        0.001804     0.014861     -0.013118       1.294744                     0.187741     -0.125026
           0.10           52           52             0  0.576923        0.003003     0.015796     -0.014442       1.491494                     0.156778     -0.095709
           0.15           17           17             0  0.764706        0.008017     0.015206     -0.015348       3.219798                     0.142357     -0.031607


,trade_count,long_trades,short_trades,win_rate,average_return,median_return,average_win,average_loss,profit_factor,total_net_return_compounded,total_net_return_sum,max_drawdown,experiment_name,edge_threshold,min_direction_probability,horizon_bars,take_profit_pct,stop_loss_pct,slippage_bps,fee_bps_per_side,trades_path,long_edge_threshold,short_edge_threshold
0,105,105,0,0.533333,0.001804,0.001612,0.014861,-0.013118,1.294744,0.187741,0.189450,-0.125026,transformer_hmm_features_full_long_only_min045...,0.05,0.45,12,0.99,0.99,1.0,0.0,/Users/rchbeir/Desktop/Quant work/quant_proj/d...,None,99.0
1,52,52,0,0.576923,0.003003,0.004035,0.015796,-0.014442,1.491494,0.156778,0.156162,-0.095709,transformer_hmm_features_full_long_only_min045...,0.10,0.45,12,0.99,0.99,1.0,0.0,/Users/rchbeir/Desktop/Quant work/quant_proj/d...,None,99.0
2,17,17,0,0.764706,0.008017,0.008731,0.015206,-0.015348,3.219798,0.142357,0.136281,-0.031607,transformer_hmm_features_full_long_only_min045...,0.15,0.45,12,0.99,0.99,1.0,0.0,/Users/rchbeir/Desktop/Quant work/quant_proj/d...,None,99.0


In [84]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

processed_dir = Path(PROCESSED_DATA_DIR)

base = pd.read_csv(
    processed_dir
    / "probability_backtests_5min"
    / "continued_transformer_long_only_min045_timeout"
    / "backtest_summary.csv"
)

hmm = pd.read_csv(
    processed_dir
    / "probability_backtests_5min"
    / "transformer_hmm_features_full_long_only_min045_timeout"
    / "backtest_summary.csv"
)

base["model"] = "base_continued_transformer"
hmm["model"] = "hmm_feature_transformer"

comparison = pd.concat([base, hmm], ignore_index=True)

display(
    comparison[
        [
            "model",
            "edge_threshold",
            "trade_count",
            "win_rate",
            "average_return",
            "profit_factor",
            "total_net_return_compounded",
            "max_drawdown",
        ]
    ]
    .sort_values(
        ["edge_threshold", "total_net_return_compounded"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

,model,edge_threshold,trade_count,win_rate,average_return,profit_factor,total_net_return_compounded,max_drawdown
0,hmm_feature_transformer,0.05,105,0.533333,0.001804,1.294744,0.187741,-0.125026
1,base_continued_transformer,0.05,138,0.536232,0.001214,1.177268,0.154416,-0.211124
2,base_continued_transformer,0.10,78,0.641026,0.003343,1.600391,0.281835,-0.065727
3,hmm_feature_transformer,0.10,52,0.576923,0.003003,1.491494,0.156778,-0.095709
4,hmm_feature_transformer,0.15,17,0.764706,0.008017,3.219798,0.142357,-0.031607
5,base_continued_transformer,0.15,32,0.625000,0.003876,1.837761,0.127076,-0.057404


In [85]:
BEST_HMM_BALANCED = {
    "edge_threshold": 0.05,
    "min_direction_probability": 0.45,
    "trade_count": 105,
    "profit_factor": 1.2947,
    "compounded_return": 0.1877,
    "max_drawdown": -0.1250,
}

BEST_HMM_CONSERVATIVE = {
    "edge_threshold": 0.15,
    "min_direction_probability": 0.45,
    "trade_count": 17,
    "profit_factor": 3.2198,
    "compounded_return": 0.1424,
    "max_drawdown": -0.0316,
}

In [86]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

trades_path = (
    Path(PROCESSED_DATA_DIR)
    / "probability_backtests_5min"
    / "transformer_hmm_features_full_long_only_min045_timeout"
    / "trades_edge_0p050.parquet"
)

trades = pd.read_parquet(trades_path)

fold_summary = (
    trades
    .groupby("fold")["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        avg_return="mean",
        total_return_sum="sum",
        compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)

display(fold_summary)

print("Positive folds:")
print(
    (fold_summary["total_return_sum"] > 0).sum(),
    "out of",
    len(fold_summary),
)

,fold,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,fold_01,5,0.600000,-0.003037,-0.015184,-0.015474
1,fold_02,21,0.761905,0.007920,0.166317,0.173503
2,fold_03,43,0.395349,-0.001377,-0.059214,-0.063263
3,fold_04,5,0.600000,0.003165,0.015824,0.015026
4,fold_05,18,0.555556,-0.000338,-0.006086,-0.007877
5,fold_06,2,1.000000,0.027129,0.054257,0.054964
6,fold_07,1,0.000000,-0.013891,-0.013891,-0.013891
7,fold_09,1,0.000000,-0.001954,-0.001954,-0.001954
8,fold_11,9,0.555556,0.005487,0.049381,0.049633


Positive folds:
4 out of 9


In [87]:
trades_path = (
    Path(PROCESSED_DATA_DIR)
    / "probability_backtests_5min"
    / "transformer_hmm_features_full_long_only_min045_timeout"
    / "trades_edge_0p150.parquet"
)

trades = pd.read_parquet(trades_path)

fold_summary = (
    trades
    .groupby("fold")["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        avg_return="mean",
        total_return_sum="sum",
        compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)

display(fold_summary)

print("Positive folds:")
print(
    (fold_summary["total_return_sum"] > 0).sum(),
    "out of",
    len(fold_summary),
)

,fold,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,fold_02,2,0.000000,-0.010676,-0.021352,-0.021311
1,fold_03,7,0.857143,0.014688,0.102814,0.106304
2,fold_04,1,1.000000,0.016334,0.016334,0.016334
3,fold_05,6,0.833333,0.002706,0.016238,0.015525
4,fold_06,1,1.000000,0.022247,0.022247,0.022247


Positive folds:
4 out of 5


In [90]:
from pathlib import Path
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.regime.hmm_feature_builder as hmm_feature_builder

importlib.reload(hmm_feature_builder)

hmm_locked_split_dir = (
    hmm_feature_builder.build_hmm_augmented_locked_test_split(
        split_name="locked_test_transformer_split_5min",
        output_name="locked_test_transformer_split_hmm_5min",
        processed_dir=PROCESSED_DATA_DIR,
        number_of_states=3,
        random_seed=42,
        covariance_type="diag",
    )
)

print(hmm_locked_split_dir)

Saved HMM-augmented locked-test split to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/locked_test_transformer_split_hmm_5min
/Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/locked_test_transformer_split_hmm_5min


In [91]:
from pathlib import Path
import importlib

from quant_research.features.feature_builder import MODEL_FEATURES
from quant_research.regime.hmm_feature_builder import HMM_OUTPUT_FEATURES_3_STATE
from quant_research.utils.paths import PROCESSED_DATA_DIR

import quant_research.models.train_transformer_locked_test as train_transformer_locked_test

importlib.reload(train_transformer_locked_test)

MODEL_FEATURES_HMM = MODEL_FEATURES + HMM_OUTPUT_FEATURES_3_STATE

print("Base features:", len(MODEL_FEATURES))
print("HMM features:", len(HMM_OUTPUT_FEATURES_3_STATE))
print("Total features:", len(MODEL_FEATURES_HMM))
print("Module path:", train_transformer_locked_test.__file__)

hmm_locked_summary = (
    train_transformer_locked_test.train_final_transformer_locked_test(
        experiment_name="final_transformer_hmm_seq21_long_candidate",

        timeframe="5min",
        processed_dir=PROCESSED_DATA_DIR,

        # Use the HMM-augmented split from Cell 1.
        split_name="locked_test_transformer_split_hmm_5min",
        recreate_split=False,

        sequence_length=21,
        batch_size=256,

        d_model=64,
        num_heads=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.10,

        learning_rate=5e-5,
        weight_decay=1e-6,

        max_epochs=30,
        patience=5,

        continue_training=True,
        continue_learning_rate=2e-5,
        continue_weight_decay=1e-5,
        continue_epochs=10,
        continue_patience=3,

        device=None,
        random_seed=42,

        feature_columns=MODEL_FEATURES_HMM,
    )
)

display(hmm_locked_summary)

Base features: 110
HMM features: 6
Total features: 116
Module path: /Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/train_transformer_locked_test.py
Training locked-test model with 116 features
Split directory: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/locked_test_transformer_split_hmm_5min
Output directory: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/transformer_locked_test_5min/final_transformer_hmm_seq21_long_candidate


/Users/rchbeir/Desktop/Quant work/quant_proj/src/quant_research/models/neural_models.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


stage_1 epoch 1: train_loss=0.7648, early_loss=0.8276, macro_f1=0.4529
stage_1 epoch 2: train_loss=0.7095, early_loss=0.8037, macro_f1=0.4422
stage_1 epoch 3: train_loss=0.7012, early_loss=0.7957, macro_f1=0.4281
stage_1 epoch 4: train_loss=0.6966, early_loss=0.8006, macro_f1=0.4329
stage_1 epoch 5: train_loss=0.6941, early_loss=0.7952, macro_f1=0.4300
stage_1 epoch 6: train_loss=0.6915, early_loss=0.7961, macro_f1=0.4160
stage_1 epoch 7: train_loss=0.6890, early_loss=0.7925, macro_f1=0.4166
stage_1 epoch 8: train_loss=0.6866, early_loss=0.7872, macro_f1=0.4107
stage_1 epoch 9: train_loss=0.6834, early_loss=0.8022, macro_f1=0.4255
stage_1 epoch 10: train_loss=0.6812, early_loss=0.7981, macro_f1=0.4146
stage_1 epoch 11: train_loss=0.6790, early_loss=0.7997, macro_f1=0.4173
stage_1 epoch 12: train_loss=0.6756, early_loss=0.7937, macro_f1=0.4159
stage_1 epoch 13: train_loss=0.6742, early_loss=0.7922, macro_f1=0.4178
stage_2_continue epoch 1: train_loss=0.6834, early_loss=0.7928, macro_f1=

,experiment_name,best_early_stop_loss,stage_1_best_epoch,stage_2_continued_best_epoch,train_sequences,early_stop_sequences,locked_test_sequences,sequence_length,num_features,learning_rate,weight_decay,continue_training,continue_learning_rate,continue_weight_decay,device,accuracy,balanced_accuracy,macro_f1,weighted_f1,log_loss,multiclass_brier_score
0,final_transformer_hmm_seq21_long_candidate,0.787174,8,0,60861,6819,16936,21,116,0.00005,0.000001,True,0.00002,0.00001,mps,0.792277,0.429159,0.442466,0.741316,0.570381,0.306219


In [92]:
from pathlib import Path
import importlib

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.backtesting.probability_backtester as probability_backtester

importlib.reload(probability_backtester)

hmm_locked_backtest = probability_backtester.run_probability_backtest_sweep(
    experiment_name="final_hmm_transformer_locked_test_long_only_min045_edge015",
    results_dir=(
        Path(PROCESSED_DATA_DIR)
        / "transformer_locked_test_5min"
        / "final_transformer_hmm_seq21_long_candidate"
    ),
    output_dir=(
        Path(PROCESSED_DATA_DIR)
        / "probability_backtests_5min"
        / "final_hmm_transformer_locked_test_long_only_min045_edge015"
    ),

    timeframe="5min",
    processed_dir=PROCESSED_DATA_DIR,

    edge_thresholds=[0.15],
    long_edge_threshold=None,
    short_edge_threshold=99.0,

    min_direction_probability=0.45,
    horizon_bars=12,

    take_profit_pct=0.99,
    stop_loss_pct=0.99,

    slippage_bps=1.0,
    fee_bps_per_side=0.0,
)

display(hmm_locked_backtest)

Experiment: final_hmm_transformer_locked_test_long_only_min045_edge015
Saved backtest summary to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/probability_backtests_5min/final_hmm_transformer_locked_test_long_only_min045_edge015/backtest_summary.csv

Backtest summary:
 edge_threshold  trade_count  long_trades  short_trades  win_rate  average_return  average_win  average_loss  profit_factor  total_net_return_compounded  max_drawdown
           0.15           16           16             0     0.625        0.008835     0.022556     -0.014034         2.6787                     0.145702     -0.071519


,trade_count,long_trades,short_trades,win_rate,average_return,median_return,average_win,average_loss,profit_factor,total_net_return_compounded,total_net_return_sum,max_drawdown,experiment_name,edge_threshold,min_direction_probability,horizon_bars,take_profit_pct,stop_loss_pct,slippage_bps,fee_bps_per_side,trades_path,long_edge_threshold,short_edge_threshold
0,16,16,0,0.625,0.008835,0.004433,0.022556,-0.014034,2.6787,0.145702,0.141353,-0.071519,final_hmm_transformer_locked_test_long_only_mi...,0.15,0.45,12,0.99,0.99,1.0,0.0,/Users/rchbeir/Desktop/Quant work/quant_proj/d...,None,99.0


In [93]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

processed_dir = Path(PROCESSED_DATA_DIR)

base_locked = pd.read_csv(
    processed_dir
    / "probability_backtests_5min"
    / "final_transformer_locked_test_long_only_min045_edge010"
    / "backtest_summary.csv"
)

hmm_locked = pd.read_csv(
    processed_dir
    / "probability_backtests_5min"
    / "final_hmm_transformer_locked_test_long_only_min045_edge015"
    / "backtest_summary.csv"
)

base_locked["model"] = "base_transformer_locked_edge010"
hmm_locked["model"] = "hmm_transformer_locked_edge015"

locked_comparison = pd.concat(
    [base_locked, hmm_locked],
    ignore_index=True,
)

display(
    locked_comparison[
        [
            "model",
            "edge_threshold",
            "trade_count",
            "win_rate",
            "average_return",
            "profit_factor",
            "total_net_return_compounded",
            "max_drawdown",
        ]
    ]
)

,model,edge_threshold,trade_count,win_rate,average_return,profit_factor,total_net_return_compounded,max_drawdown
0,base_transformer_locked_edge010,0.10,33,0.484848,0.000828,1.121513,0.022348,-0.071643
1,hmm_transformer_locked_edge015,0.15,16,0.625000,0.008835,2.678700,0.145702,-0.071519


In [94]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

trades_path = (
    Path(PROCESSED_DATA_DIR)
    / "probability_backtests_5min"
    / "final_hmm_transformer_locked_test_long_only_min045_edge015"
    / "trades_edge_0p150.parquet"
)

hmm_locked_trades = pd.read_parquet(trades_path)

hmm_locked_trades["entry_timestamp_pt"] = pd.to_datetime(
    hmm_locked_trades["entry_timestamp_pt"]
)

hmm_locked_trades["month"] = (
    hmm_locked_trades["entry_timestamp_pt"]
    .dt.tz_localize(None)
    .dt.to_period("M")
)

monthly_summary = (
    hmm_locked_trades
    .groupby("month")["net_return"]
    .agg(
        trade_count="count",
        win_rate=lambda x: (x > 0).mean(),
        avg_return="mean",
        total_return_sum="sum",
        compounded_return=lambda x: (1 + x).prod() - 1,
    )
    .reset_index()
)

display(monthly_summary)

print("Positive months:")
print(
    (monthly_summary["total_return_sum"] > 0).sum(),
    "out of",
    len(monthly_summary),
)

display(
    hmm_locked_trades[
        [
            "date",
            "side",
            "net_return",
            "p_neutral",
            "p_up",
            "p_down",
            "signal_edge",
            "target_name",
            "bars_held",
        ]
    ].sort_values("net_return")
)

,month,trade_count,win_rate,avg_return,total_return_sum,compounded_return
0,2025-02,1,1.0,0.003983,0.003983,0.003983
1,2025-04,15,0.6,0.009158,0.137370,0.141156


Positive months:
2 out of 2


,date,side,net_return,p_neutral,p_up,p_down,signal_edge,target_name,bars_held
5,2025-04-08,long,-0.034793,0.091772,0.530848,0.377381,0.153467,up,12
8,2025-04-08,long,-0.017920,0.083992,0.543518,0.372490,0.171029,down,12
7,2025-04-08,long,-0.012992,0.051843,0.605953,0.342205,0.263748,down,12
13,2025-04-10,long,-0.010880,0.064218,0.548838,0.386944,0.161894,up,12
9,2025-04-09,long,-0.004130,0.049160,0.557240,0.393599,0.163641,down,12
6,2025-04-08,long,-0.003488,0.049415,0.561536,0.389049,0.172487,down,12
2,2025-04-07,long,0.003971,0.057036,0.576608,0.366356,0.210253,up,12
0,2025-02-25,long,0.003983,0.156186,0.503889,0.339924,0.163965,neutral,12
1,2025-04-04,long,0.004883,0.042290,0.555575,0.402135,0.153440,up,12
15,2025-04-10,long,0.005818,0.062650,0.556650,0.380699,0.175951,down,12


In [95]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

pred_path = (
    Path(PROCESSED_DATA_DIR)
    / "transformer_locked_test_5min"
    / "final_transformer_hmm_seq21_long_candidate"
    / "fold_01"
    / "validation_predictions.parquet"
)

pred = pd.read_parquet(pred_path)

pred["timestamp_pt"] = pd.to_datetime(pred["timestamp_pt"])
pred["month"] = pred["timestamp_pt"].dt.tz_localize(None).dt.to_period("M")

pred["long_edge"] = pred["p_up"] - pred["p_down"]

for edge in [0.05, 0.10, 0.15]:
    mask = (
        (pred["p_up"] >= 0.45)
        & (pred["long_edge"] >= edge)
    )

    monthly = (
        pred.loc[mask]
        .groupby("month")
        .size()
        .reset_index(name=f"signals_edge_{edge}")
    )

    print(f"\nLong signal counts, edge={edge}")
    display(monthly)


Long signal counts, edge=0.05


,month,signals_edge_0.05
0,2024-10,1
1,2024-11,5
2,2025-01,12
3,2025-02,12
4,2025-03,52
5,2025-04,235
6,2025-05,2
7,2025-10,2
8,2025-11,20



Long signal counts, edge=0.1


,month,signals_edge_0.1
0,2025-01,3
1,2025-02,3
2,2025-03,16
3,2025-04,157
4,2025-05,1
5,2025-10,1
6,2025-11,9



Long signal counts, edge=0.15


,month,signals_edge_0.15
0,2025-02,1
1,2025-04,80


In [96]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR

processed_dir = Path(PROCESSED_DATA_DIR)

base_pred_path = (
    processed_dir
    / "transformer_locked_test_5min"
    / "final_transformer_seq21_long_candidate"
    / "fold_01"
    / "validation_predictions.parquet"
)

hmm_pred_path = (
    processed_dir
    / "transformer_locked_test_5min"
    / "final_transformer_hmm_seq21_long_candidate"
    / "fold_01"
    / "validation_predictions.parquet"
)

base = pd.read_parquet(base_pred_path)
hmm = pd.read_parquet(hmm_pred_path)

for df in [base, hmm]:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["timestamp_pt"] = pd.to_datetime(df["timestamp_pt"])
    df["month"] = df["timestamp_pt"].dt.tz_localize(None).dt.to_period("M")

base["base_long_edge"] = base["p_up"] - base["p_down"]
hmm["hmm_long_edge"] = hmm["p_up"] - hmm["p_down"]

base_signals = base[
    (base["p_up"] >= 0.45)
    & (base["base_long_edge"] >= 0.10)
][["timestamp", "timestamp_pt", "month", "p_up", "p_down", "base_long_edge"]].copy()

hmm_signals = hmm[
    (hmm["p_up"] >= 0.45)
    & (hmm["hmm_long_edge"] >= 0.15)
][["timestamp", "timestamp_pt", "month", "p_up", "p_down", "hmm_long_edge"]].copy()

overlap = base_signals.merge(
    hmm_signals,
    on=["timestamp", "timestamp_pt", "month"],
    how="inner",
    suffixes=("_base", "_hmm"),
)

print("Base signals:", len(base_signals))
print("HMM signals:", len(hmm_signals))
print("Overlap signals:", len(overlap))

print("\nBase signal months:")
display(base_signals.groupby("month").size().reset_index(name="base_signals"))

print("\nHMM signal months:")
display(hmm_signals.groupby("month").size().reset_index(name="hmm_signals"))

print("\nOverlap signal months:")
display(overlap.groupby("month").size().reset_index(name="overlap_signals"))

display(overlap.head())

Base signals: 159
HMM signals: 81
Overlap signals: 60

Base signal months:


,month,base_signals
0,2024-11,1
1,2024-12,2
2,2025-01,8
3,2025-02,9
4,2025-03,24
5,2025-04,109
6,2025-10,1
7,2025-11,5



HMM signal months:


,month,hmm_signals
0,2025-02,1
1,2025-04,80



Overlap signal months:


,month,overlap_signals
0,2025-02,1
1,2025-04,59


,timestamp,timestamp_pt,month,p_up_base,p_down_base,base_long_edge,p_up_hmm,p_down_hmm,hmm_long_edge
0,2025-02-25 15:45:00+00:00,2025-02-25 07:45:00-08:00,2025-02,0.482206,0.342274,0.139932,0.503889,0.339924,0.163965
1,2025-04-04 14:55:00+00:00,2025-04-04 07:55:00-07:00,2025-04,0.539072,0.389718,0.149355,0.555575,0.402135,0.153440
2,2025-04-04 15:00:00+00:00,2025-04-04 08:00:00-07:00,2025-04,0.527925,0.406014,0.121911,0.542875,0.382475,0.160400
3,2025-04-04 15:10:00+00:00,2025-04-04 08:10:00-07:00,2025-04,0.530551,0.397776,0.132774,0.561968,0.392326,0.169642
4,2025-04-04 15:15:00+00:00,2025-04-04 08:15:00-07:00,2025-04,0.548362,0.362653,0.185709,0.570877,0.385471,0.185407


In [97]:
from pathlib import Path
import pandas as pd

from quant_research.utils.paths import PROCESSED_DATA_DIR
import quant_research.backtesting.probability_backtester as probability_backtester

processed_dir = Path(PROCESSED_DATA_DIR)

base_pred_path = (
    processed_dir
    / "transformer_locked_test_5min"
    / "final_transformer_seq21_long_candidate"
    / "fold_01"
    / "validation_predictions.parquet"
)

hmm_pred_path = (
    processed_dir
    / "transformer_locked_test_5min"
    / "final_transformer_hmm_seq21_long_candidate"
    / "fold_01"
    / "validation_predictions.parquet"
)

base = pd.read_parquet(base_pred_path)
hmm = pd.read_parquet(hmm_pred_path)

base["timestamp"] = pd.to_datetime(base["timestamp"], utc=True)
hmm["timestamp"] = pd.to_datetime(hmm["timestamp"], utc=True)

base["base_long_edge"] = base["p_up"] - base["p_down"]
hmm["hmm_long_edge"] = hmm["p_up"] - hmm["p_down"]

base_keep = base[
    [
        "timestamp",
        "timestamp_pt",
        "date",
        "target_name",
        "target_class",
        "p_neutral",
        "p_up",
        "p_down",
        "base_long_edge",
    ]
].copy()

hmm_keep = hmm[
    [
        "timestamp",
        "hmm_long_edge",
        "p_up",
        "p_down",
    ]
].copy()

hmm_keep = hmm_keep.rename(
    columns={
        "p_up": "hmm_p_up",
        "p_down": "hmm_p_down",
    }
)

combined = base_keep.merge(
    hmm_keep,
    on="timestamp",
    how="inner",
)

overlap_mask = (
    (combined["p_up"] >= 0.45)
    & (combined["base_long_edge"] >= 0.10)
    & (combined["hmm_p_up"] >= 0.45)
    & (combined["hmm_long_edge"] >= 0.15)
)

overlap_predictions = combined.loc[overlap_mask].copy()

# Make the backtester think these are high-confidence long predictions.
# p_down is lowered so p_up - p_down clears the threshold.
overlap_predictions["p_neutral"] = 0.05
overlap_predictions["p_up"] = 0.90
overlap_predictions["p_down"] = 0.05

# Save in the same structure expected by the existing backtester.
overlap_results_dir = (
    processed_dir
    / "transformer_locked_test_5min"
    / "final_overlap_base_hmm_locked"
    / "fold_01"
)

overlap_results_dir.mkdir(parents=True, exist_ok=True)

overlap_predictions.to_parquet(
    overlap_results_dir / "validation_predictions.parquet",
    index=False,
)

print("Overlap prediction rows saved:", len(overlap_predictions))
display(overlap_predictions.head())

overlap_backtest = probability_backtester.run_probability_backtest_sweep(
    experiment_name="final_overlap_base_hmm_locked_long_only",
    results_dir=(
        processed_dir
        / "transformer_locked_test_5min"
        / "final_overlap_base_hmm_locked"
    ),
    output_dir=(
        processed_dir
        / "probability_backtests_5min"
        / "final_overlap_base_hmm_locked_long_only"
    ),

    timeframe="5min",
    processed_dir=PROCESSED_DATA_DIR,

    edge_thresholds=[0.10],
    long_edge_threshold=None,
    short_edge_threshold=99.0,

    min_direction_probability=0.45,
    horizon_bars=12,

    take_profit_pct=0.99,
    stop_loss_pct=0.99,

    slippage_bps=1.0,
    fee_bps_per_side=0.0,
)

display(overlap_backtest)

Overlap prediction rows saved: 60


,timestamp,timestamp_pt,date,target_name,target_class,p_neutral,p_up,p_down,base_long_edge,hmm_long_edge,hmm_p_up,hmm_p_down
5119,2025-02-25 15:45:00+00:00,2025-02-25 07:45:00-08:00,2025-02-25,neutral,0,0.05,0.9,0.05,0.139932,0.163965,0.503889,0.339924
6661,2025-04-04 14:55:00+00:00,2025-04-04 07:55:00-07:00,2025-04-04,up,1,0.05,0.9,0.05,0.149355,0.153440,0.555575,0.402135
6662,2025-04-04 15:00:00+00:00,2025-04-04 08:00:00-07:00,2025-04-04,down,2,0.05,0.9,0.05,0.121911,0.160400,0.542875,0.382475
6664,2025-04-04 15:10:00+00:00,2025-04-04 08:10:00-07:00,2025-04-04,down,2,0.05,0.9,0.05,0.132774,0.169642,0.561968,0.392326
6665,2025-04-04 15:15:00+00:00,2025-04-04 08:15:00-07:00,2025-04-04,up,1,0.05,0.9,0.05,0.185709,0.185407,0.570877,0.385471


Experiment: final_overlap_base_hmm_locked_long_only
Saved backtest summary to: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/probability_backtests_5min/final_overlap_base_hmm_locked_long_only/backtest_summary.csv

Backtest summary:
 edge_threshold  trade_count  long_trades  short_trades  win_rate  average_return  average_win  average_loss  profit_factor  total_net_return_compounded  max_drawdown
            0.1           14           14             0  0.642857        0.004816     0.014856     -0.013257       2.017121                     0.067179     -0.054577


,trade_count,long_trades,short_trades,win_rate,average_return,median_return,average_win,average_loss,profit_factor,total_net_return_compounded,total_net_return_sum,max_drawdown,experiment_name,edge_threshold,min_direction_probability,horizon_bars,take_profit_pct,stop_loss_pct,slippage_bps,fee_bps_per_side,trades_path,long_edge_threshold,short_edge_threshold
0,14,14,0,0.642857,0.004816,0.004433,0.014856,-0.013257,2.017121,0.067179,0.067419,-0.054577,final_overlap_base_hmm_locked_long_only,0.1,0.45,12,0.99,0.99,1.0,0.0,/Users/rchbeir/Desktop/Quant work/quant_proj/d...,None,99.0


In [98]:
base_locked = pd.read_csv(
    processed_dir
    / "probability_backtests_5min"
    / "final_transformer_locked_test_long_only_min045_edge010"
    / "backtest_summary.csv"
)

hmm_locked = pd.read_csv(
    processed_dir
    / "probability_backtests_5min"
    / "final_hmm_transformer_locked_test_long_only_min045_edge015"
    / "backtest_summary.csv"
)

overlap_locked = pd.read_csv(
    processed_dir
    / "probability_backtests_5min"
    / "final_overlap_base_hmm_locked_long_only"
    / "backtest_summary.csv"
)

base_locked["model"] = "base_only"
hmm_locked["model"] = "hmm_only"
overlap_locked["model"] = "base_and_hmm_overlap"

final_compare = pd.concat(
    [base_locked, hmm_locked, overlap_locked],
    ignore_index=True,
)

display(
    final_compare[
        [
            "model",
            "edge_threshold",
            "trade_count",
            "win_rate",
            "average_return",
            "profit_factor",
            "total_net_return_compounded",
            "max_drawdown",
        ]
    ]
)

,model,edge_threshold,trade_count,win_rate,average_return,profit_factor,total_net_return_compounded,max_drawdown
0,base_only,0.10,33,0.484848,0.000828,1.121513,0.022348,-0.071643
1,hmm_only,0.15,16,0.625000,0.008835,2.678700,0.145702,-0.071519
2,base_and_hmm_overlap,0.10,14,0.642857,0.004816,2.017121,0.067179,-0.054577


## Final Strategy Status

The transformer and HMM experiments found evidence of regime-dependent predictive structure, but not enough evidence for a robust standalone trading strategy.

The base transformer produced broader locked-test coverage, but only a weak positive result: 33 trades, +2.23% compounded return, 1.12 profit factor, and -7.16% max drawdown.

The HMM-feature transformer produced stronger locked-test performance, but the trades were too sparse and highly concentrated in April 2025. The HMM-only rule had 16 trades and +14.57% compounded return, while the base/HMM overlap rule had 14 trades and +6.72% compounded return. These results are promising, but not statistically reliable because most of the evidence comes from one regime cluster.

Conclusion: this is not yet a deployable strategy. The current models should be treated as research signals rather than production trading systems. The HMM appears useful as a regime-specific confidence feature, but more out-of-sample evidence, broader market coverage, and stronger regime generalization are needed before calling it a real strategy.